# Olist E-Commerce Analysis — Data Cleaning and Preparation

## Objective

The purpose of this notebook is to implement the cleaning decisions documented during the data-quality audit and produce reliable, reusable datasets for analysis.

The raw CSV files will remain unchanged. Cleaned copies of the required tables will be created, validated, and saved in the `data/processed` folder.

The cleaning process will include:

* Converting timestamp columns to datetime format.
* Standardizing ZIP-code prefixes as five-character strings.
* Correcting documented column-name and category-translation issues.
* Handling missing categories and invalid numeric values.
* Creating flags for timestamp, payment, review, product, and geographic exceptions.
* Aggregating one-to-many tables before analytical joins.
* Validating row counts, keys, missing values, and data types after cleaning.
* Saving processed tables without overwriting the raw source files.

Records will not be deleted or imputed without a documented reason. When a record remains valid for some analyses but not others, an exception flag will be created so it can be retained and selectively excluded.


## 1. Setup and Raw Data Loading

The raw Olist CSV files will be loaded into clearly named DataFrames. Variables ending in `_raw` will represent unchanged source data, while separate variables ending in `_clean` will be created for cleaning. This naming convention prevents accidental modification of the audit source.


In [ ]:
import pandas as pd
from pathlib import Path


Raw_Data_Path = Path("../data/raw")
Processed_Data_Path = Path("../data/processed")

print(Raw_Data_Path.resolve())
print(Processed_Data_Path.resolve())

C:\Users\Hp\DataAnalyst\data\raw
C:\Users\Hp\DataAnalyst\data\processed


In [ ]:
#loading data
orders_raw = pd.read_csv(Raw_Data_Path/"olist_orders_dataset.csv")
customers_raw = pd.read_csv(Raw_Data_Path/"olist_customers_dataset.csv")
order_items_raw = pd.read_csv(Raw_Data_Path/"olist_order_items_dataset.csv")
payments_raw = pd.read_csv(Raw_Data_Path/"olist_order_payments_dataset.csv")
reviews_raw = pd.read_csv(Raw_Data_Path/'olist_order_reviews_dataset.csv')
products_raw = pd.read_csv(Raw_Data_Path/'olist_products_dataset.csv')
translation_raw = pd.read_csv(Raw_Data_Path/'product_category_name_translation.csv')
sellers_raw = pd.read_csv(Raw_Data_Path/'olist_sellers_dataset.csv')
geolocation_raw = pd.read_csv(Raw_Data_Path/'olist_geolocation_dataset.csv')

In [ ]:
print(f"orders_raw shape: {orders_raw.shape}")
print(f"customers_raw shape: {customers_raw.shape}")
print(f"order_items_raw shape: {order_items_raw.shape}")
print(f"payments_raw shape: {payments_raw.shape}")
print(f"reviews_raw shape: {reviews_raw.shape}")
print(f"products_raw shape: {products_raw.shape}")
print(f"translation_raw shape: {translation_raw.shape}")
print(f"sellers_raw shape: {sellers_raw.shape}")
print(f"geolocation_raw shape: {geolocation_raw.shape}")

orders_raw shape: (99441, 8)
customers_raw shape: (99441, 5)
order_items_raw shape: (112650, 7)
payments_raw shape: (103886, 5)
reviews_raw shape: (99224, 7)
products_raw shape: (32951, 9)
translation_raw shape: (71, 2)
sellers_raw shape: (3095, 4)
geolocation_raw shape: (1000163, 5)


## 2. Orders Cleaning

A clean copy of the Orders table will be created while preserving all raw records. Timestamp columns will be converted to datetime, and documented data-quality exceptions will be represented using Boolean flags rather than deleting orders.

The cleaning steps will include:

* Converting five timestamp columns to datetime.
* Confirming that conversion introduces no new missing values.
* Flagging delivered orders with missing approval, carrier-delivery, or customer-delivery timestamps.
* Flagging inconsistent lifecycle timestamp sequences.
* Preserving valid order identifiers and status values.
* Validating row count, key uniqueness, data types, and exception counts after cleaning.

### 2.1 Create the Clean Orders Copy

The raw Orders DataFrame will be copied before any transformations so that the source data remains unchanged.


In [ ]:
#Create orders_clean as an independent copy of orders_raw
orders_clean = orders_raw.copy()

In [ ]:
if orders_raw.shape==orders_clean.shape:
    print("Both orders_raw and orders_clean df's have same shapes")
    print(f"orders_raw shape: {orders_raw.shape}")
    print(f"orders_clean shape: {orders_clean.shape}")
else:
    print("orders_raw and orders_clean df's have different shapes")
    print(f"orders_raw shape: {orders_raw.shape}")
    print(f"orders_clean shape: {orders_clean.shape}")

Both orders_raw and orders_clean df's have same shapes
orders_raw shape: (99441, 8)
orders_clean shape: (99441, 8)


### 2.2 Timestamp Conversion

The five order lifecycle timestamp columns will be converted from strings to datetime format. Missing-value counts will be recorded before and after conversion to confirm that invalid strings do not create additional missing timestamps.


In [ ]:
orders_clean.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

In [ ]:
orders_change_timestamp = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

In [ ]:
orders_timestamp_missing_values_count_before_conversion = orders_raw[orders_change_timestamp].isna().sum()
orders_timestamp_missing_values_count_before_conversion

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [ ]:
for cols in orders_change_timestamp:
    orders_clean[cols] = pd.to_datetime(orders_clean[cols], errors="coerce")

In [ ]:
orders_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [ ]:
orders_timestamp_missing_values_count_after_conversion=orders_clean[orders_change_timestamp].isna().sum()
orders_timestamp_missing_values_count_after_conversion

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [ ]:
order_clean_timestamp_missing_comparison_summary = pd.DataFrame({"before": orders_timestamp_missing_values_count_before_conversion, "after":orders_timestamp_missing_values_count_after_conversion})
order_clean_timestamp_missing_comparison_summary

,before,after
order_purchase_timestamp,0,0
order_approved_at,160,160
order_delivered_carrier_date,1783,1783
order_delivered_customer_date,2965,2965
order_estimated_delivery_date,0,0


#### Timestamp Conversion Findings

All five order timestamp columns were successfully converted from string format to datetime format. The missing-value counts remained unchanged before and after conversion, confirming that no valid timestamp values were lost or converted to `NaT` because of parsing errors.

The existing missing timestamps will be retained because their treatment depends on the corresponding order status. Delivered orders with missing critical timestamps will be flagged as data-quality exceptions instead of being removed or imputed.

### 2.3 Missing Timestamp Exception Flags

Boolean flags will be created to identify delivered orders that are missing approval, carrier-delivery, or customer-delivery timestamps. These flags will preserve the affected orders for completed-sales analysis while allowing them to be excluded from calculations that require the missing timestamps.


In [ ]:
# Create a Boolean mask for delivered orders with a missing order_delivered_customer_date timestamp

orders_clean_missing_delivered_ts = (orders_clean['order_status']=='delivered') & (orders_clean['order_delivered_customer_date'].isna()) 
orders_clean_missing_delivered_ts

0        False
1        False
2        False
3        False
4        False
         ...  
99436    False
99437    False
99438    False
99439    False
99440    False
Length: 99441, dtype: bool

In [ ]:
# Add the missing-customer date timestamp flag to orders_clean
orders_clean['flag_delivered_missing_customer_date'] = orders_clean_missing_delivered_ts
orders_clean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,flag_delivered_missing_customer_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False


In [ ]:
# Count delivered orders with a missing customer-delivery date

len(orders_clean[orders_clean['flag_delivered_missing_customer_date'] == True ])

8

In [ ]:
# Create a Boolean mask for delivered orders with a missing order_delivered_carrier_date timestamp

orders_clean_missing_delivered_ts_order_delivered_carrier_date = (orders_clean['order_status']=='delivered') & (orders_clean['order_delivered_carrier_date'].isna()) 
orders_clean_missing_delivered_ts_order_delivered_carrier_date

0        False
1        False
2        False
3        False
4        False
         ...  
99436    False
99437    False
99438    False
99439    False
99440    False
Length: 99441, dtype: bool

In [ ]:
# Add the missing-carrier date timestamp flag to orders_clean
orders_clean['flag_delivered_missing_carrier_date'] = orders_clean_missing_delivered_ts_order_delivered_carrier_date
orders_clean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,flag_delivered_missing_customer_date,flag_delivered_missing_carrier_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False


In [ ]:
# Count delivered orders with a missing carrier-delivery date
len(orders_clean[orders_clean['flag_delivered_missing_carrier_date'] == True ])

2

In [ ]:
# Create a Boolean mask for delivered orders with a missing approval timestamp
orders_clean_missing_delivered_ts_order_approved_at = (orders_clean['order_status']=='delivered') & (orders_clean['order_approved_at'].isna()) 
orders_clean_missing_delivered_ts_order_approved_at

0        False
1        False
2        False
3        False
4        False
         ...  
99436    False
99437    False
99438    False
99439    False
99440    False
Length: 99441, dtype: bool

In [ ]:
# Add the missing-approval timestamp flag to orders_clean
orders_clean['flag_delivered_missing_approval_date'] = orders_clean_missing_delivered_ts_order_approved_at
orders_clean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,flag_delivered_missing_customer_date,flag_delivered_missing_carrier_date,flag_delivered_missing_approval_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,False


In [ ]:
# Count delivered orders with a missing approval timestamp
len(orders_clean[orders_clean['flag_delivered_missing_approval_date'] == True ])

14

In [ ]:
# Create a combined flag for delivered orders missing any critical timestamp

bool_delivered_missing_critical_timestamp = (orders_clean['flag_delivered_missing_customer_date'] == True) | (orders_clean['flag_delivered_missing_carrier_date'] == True) | (orders_clean['flag_delivered_missing_approval_date'] == True)

In [ ]:
# Add combined flag column for delivered orders missing any critical timestamp
orders_clean['flag_delivered_missing_critical_timestamp'] = bool_delivered_missing_critical_timestamp
orders_clean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,flag_delivered_missing_customer_date,flag_delivered_missing_carrier_date,flag_delivered_missing_approval_date,flag_delivered_missing_critical_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,False,False


In [ ]:
# Count delivered orders missing any critical timestamp
len(orders_clean[orders_clean['flag_delivered_missing_critical_timestamp'] == True])

23

In [ ]:
#the overlapping record by finding orders where more than one of the three individual flags is True.
orders_clean['critical_timestamp_flag_count'] = orders_clean[['flag_delivered_missing_customer_date', 'flag_delivered_missing_carrier_date', 'flag_delivered_missing_approval_date']].sum(axis = 1)

#Count the overlapping record by finding orders where more than one of the three individual flags is True.
len(orders_clean[orders_clean['critical_timestamp_flag_count']>1])


1

In [ ]:
# Inspect delivered orders missing more than one critical timestamp
orders_clean[orders_clean['critical_timestamp_flag_count']>1]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,flag_delivered_missing_customer_date,flag_delivered_missing_carrier_date,flag_delivered_missing_approval_date,flag_delivered_missing_critical_timestamp,critical_timestamp_flag_count
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23,True,True,False,True,2


#### Missing Timestamp Flag Findings

Boolean flags were created for delivered orders missing critical lifecycle timestamps:

* **8 orders** are missing `order_delivered_customer_date`.
* **2 orders** are missing `order_delivered_carrier_date`.
* **14 orders** are missing `order_approved_at`.

The individual flags contain **24 flagged occurrences**, while the combined exception flag identifies **23 unique orders**. The difference occurs because one delivered order is missing both its carrier-delivery and customer-delivery timestamps.

These records will be retained for completed-sales analysis but flagged and excluded from calculations that require their missing timestamps. No timestamp values were imputed because suitable replacement values cannot be determined reliably.


### 2.4 Order Lifecycle Exception Flags

The order lifecycle timestamps will be checked for chronological inconsistencies. Boolean flags will be created for records where a later lifecycle event appears to occur before an earlier event.

The affected records will be retained rather than corrected because the dataset does not provide enough information to reconstruct the actual timestamps. These flags will allow unreliable records to be excluded from duration-based analysis.


In [ ]:
# Create a flag for orders whose carrier-delivery date is before the purchase timestamp
flag_carrier_before_purchase = (orders_clean['order_delivered_carrier_date']) < (orders_clean['order_purchase_timestamp'])
flag_carrier_before_purchase

0        False
1        False
2        False
3        False
4        False
         ...  
99436    False
99437    False
99438    False
99439    False
99440    False
Length: 99441, dtype: bool

In [ ]:
orders_clean['flag_carrier_before_purchase'] = flag_carrier_before_purchase
orders_clean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,flag_delivered_missing_customer_date,flag_delivered_missing_carrier_date,flag_delivered_missing_approval_date,flag_delivered_missing_critical_timestamp,critical_timestamp_flag_count,flag_carrier_before_purchase
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,False,False,0,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,False,False,0,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,False,False,0,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,False,False,0,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,False,False,0,False


In [ ]:
# Count orders whose carrier-delivery date occurs before purchase
int(orders_clean['flag_carrier_before_purchase'].sum())

166

In [ ]:
# Create a flag for orders delivered to the customer before the carrier-delivery date

flag_customer_before_carrier = (orders_clean['order_delivered_customer_date']) < (orders_clean['order_delivered_carrier_date'])
orders_clean['flag_customer_before_carrier'] = flag_customer_before_carrier
int(orders_clean['flag_customer_before_carrier'].sum())

23

In [ ]:
# Count orders whose carrier-delivery date occurs before approval
flag_carrier_before_approval = (orders_clean['order_delivered_carrier_date']) < (orders_clean['order_approved_at'])
orders_clean['flag_carrier_before_approval'] = flag_carrier_before_approval
int(orders_clean['flag_carrier_before_approval'].sum())

1359

In [ ]:
# Validate that no approval timestamp occurs before purchase
approval_before_purchase_check = orders_clean['order_approved_at'] < orders_clean['order_purchase_timestamp']
int(approval_before_purchase_check.sum())

0

In [ ]:
# Validate that no estimated delivery date occurs before purchase
estimated_delivery_before_purchase_check = orders_clean['order_estimated_delivery_date'] < orders_clean['order_purchase_timestamp']
int(estimated_delivery_before_purchase_check.sum())

0

In [ ]:
# Create a combined flag for orders with any lifecycle timestamp exception
flag_lifecycle_timestamp_exception = (orders_clean['flag_carrier_before_purchase'] == True) | (orders_clean['flag_customer_before_carrier'] == True) | (orders_clean['flag_carrier_before_approval'] == True)

# Add the combined lifecycle timestamp exception flag to orders_clean
orders_clean['flag_lifecycle_timestamp_exception'] = flag_lifecycle_timestamp_exception
orders_clean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,flag_delivered_missing_customer_date,flag_delivered_missing_carrier_date,flag_delivered_missing_approval_date,flag_delivered_missing_critical_timestamp,critical_timestamp_flag_count,flag_carrier_before_purchase,flag_customer_before_carrier,flag_carrier_before_approval,flag_lifecycle_timestamp_exception
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,False,False,0,False,False,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,False,False,0,False,False,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,False,False,0,False,False,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,False,False,0,False,False,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,False,False,0,False,False,False,False


In [ ]:
int(orders_clean['flag_lifecycle_timestamp_exception'].sum())

1382

#### Order Lifecycle Flag Findings

The order lifecycle sequence validation identified the following timestamp exceptions:

* **166 orders** have a carrier-delivery timestamp earlier than their purchase timestamp.
* **23 orders** have a customer-delivery timestamp earlier than their carrier-delivery timestamp.
* **1,359 orders** have a carrier-delivery timestamp earlier than their approval timestamp.
* No approval timestamps occur before the purchase timestamp.
* No estimated delivery dates occur before the purchase timestamp.

The individual exception counts total **1,548**, while the combined lifecycle flag identifies **1,382 unique orders**. This difference confirms that some orders contain more than one lifecycle inconsistency.

These records will be retained and flagged rather than corrected because the actual event timestamps cannot be reconstructed reliably. They should be excluded from calculations that depend on the affected timestamp sequence.


### 2.5 Final Orders Validation

The cleaned Orders table will be validated before it is saved to the processed-data directory. The validation will confirm that no records were unintentionally removed, the primary key remains complete and unique, timestamp columns have the correct data type, exception flags are Boolean, and the original missing-value counts remain unchanged.


In [ ]:
# Confirm that cleaning preserved the original number of order records
if orders_raw.shape[0]==orders_clean.shape[0]:
    print("Both orders_raw and orders_clean df's have same rows")
    print(f"orders_raw rows: {orders_raw.shape[0]}")
    print(f"orders_clean rows: {orders_clean.shape[0]}")
else:
    print("orders_raw and orders_clean df's have different rows")
    print(f"orders_raw rows: {orders_raw.shape[0]}")
    print(f"orders_clean rows: {orders_clean.shape[0]}")

Both orders_raw and orders_clean df's have same rows
orders_raw rows: 99441
orders_clean rows: 99441


In [ ]:
# Validate the completeness and uniqueness of the order_id primary key

print(f"Missing order_id count: {int(orders_clean['order_id'].isna().sum())}")
print(f"Duplicated order_id count: {int(orders_clean['order_id'].duplicated().sum())}")
print(f"Distinct order_id count: {orders_clean['order_id'].nunique()}")


Missing order_id count: 0
Duplicated order_id count: 0
Distinct order_id count: 99441


In [ ]:
# Confirm that all order timestamp columns use a datetime data type
orders_clean[orders_change_timestamp].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [ ]:
# Confirm that all order exception flags use the Boolean data type
flag_columns = ['flag_delivered_missing_customer_date',
'flag_delivered_missing_carrier_date',
'flag_delivered_missing_approval_date',
'flag_delivered_missing_critical_timestamp',
'flag_carrier_before_purchase',
'flag_customer_before_carrier',
'flag_carrier_before_approval',
'flag_lifecycle_timestamp_exception']

orders_clean[flag_columns].dtypes

flag_delivered_missing_customer_date         bool
flag_delivered_missing_carrier_date          bool
flag_delivered_missing_approval_date         bool
flag_delivered_missing_critical_timestamp    bool
flag_carrier_before_purchase                 bool
flag_customer_before_carrier                 bool
flag_carrier_before_approval                 bool
flag_lifecycle_timestamp_exception           bool
dtype: object

In [ ]:
# Confirm that cleaning did not alter missing values in the original Orders columns
original_columns =['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

orders_raw_missig_value_count = orders_raw[original_columns].isna().sum()
orders_clean_missig_value_count = orders_clean[original_columns].isna().sum()

Original_Columns_missing_value_summary = pd.DataFrame({"orders_raw":orders_raw_missig_value_count, "orders_clean":orders_clean_missig_value_count})
Original_Columns_missing_value_summary

,orders_raw,orders_clean
order_id,0,0
customer_id,0,0
order_status,0,0
order_purchase_timestamp,0,0
order_approved_at,160,160
order_delivered_carrier_date,1783,1783
order_delivered_customer_date,2965,2965
order_estimated_delivery_date,0,0


#### Final Orders Validation Findings

The cleaned Orders table passed all final validation checks:

* All **99,441 original order records** were preserved.
* `order_id` contains no missing or duplicated values and remains a valid primary key.
* All five lifecycle timestamp columns use a datetime data type.
* All eight exception-flag columns use the Boolean data type.
* Missing-value counts in the original Orders columns remained unchanged.
* No timestamp values were imputed or removed.
* Critical missing-timestamp and lifecycle-sequence exceptions were retained and documented through flags.

The cleaned Orders table is therefore ready to be saved and used in downstream analysis.


### 2.6 Save the Cleaned Orders Table

The validated Orders table will be saved in the processed-data directory without modifying the original raw dataset. The output will retain all original order records, converted timestamp columns, exception flags, and the critical-timestamp severity count.


In [ ]:
# Save the validated Orders table to the processed-data directory
orders_clean.to_csv(Processed_Data_Path/'olist_orders_clean.csv', index = False)

## 3. Customers Cleaning

The Customers table will be cleaned while preserving all customer records. The cleaning process will create an independent copy of the raw dataset, standardize ZIP-code prefixes as five-character strings, validate customer identifiers, and confirm the formatting of city and state values.

Repeated `customer_unique_id` values will be retained because the same customer may place multiple orders and therefore receive multiple `customer_id` values.

### 3.1 Create the Clean Customers Copy

A separate clean copy will be created so that all transformations can be performed without modifying `customers_raw`.


In [ ]:
# Create an independent working copy of the raw Customers table
customers_clean = customers_raw.copy()
customers_clean.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [ ]:
if customers_raw.shape == customers_clean.shape:
    print("Both customers_raw and customers_clean df's have same shapes")
    print(f"customers_raw shape: {customers_raw.shape}")
    print(f"customers_clean shape: {customers_clean.shape}")
else:
    print("customers_raw and customers_clean df's have different shapes")
    print(f"customers_raw shape: {customers_raw.shape}")
    print(f"customers_clean shape: {customers_clean.shape}")

Both customers_raw and customers_clean df's have same shapes
customers_raw shape: (99441, 5)
customers_clean shape: (99441, 5)


### 3.2 Customer ZIP-Code Standardization

`customer_zip_code_prefix` is currently stored as an integer. However, a ZIP-code prefix is an identifier rather than a quantity and should not be used in mathematical calculations.

The column will be converted to a string and standardized to five characters by restoring leading zeros where required. This will ensure consistent formatting and reliable joins with the Geolocation table.


In [ ]:
# Convert customer ZIP prefixes to five-character strings with leading zeros
customers_clean['customer_zip_code_prefix'] = customers_clean['customer_zip_code_prefix'].astype(str)
customers_clean['customer_zip_code_prefix'] = customers_clean['customer_zip_code_prefix'].str.zfill(5)

In [ ]:
customers_clean.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [ ]:
# Validate the format and completeness of standardized customer ZIP prefixes
print(f"length of min zip_code value: {len(min(customers_clean['customer_zip_code_prefix']))}")
print(f"length of max zip_code value: {len(max(customers_clean['customer_zip_code_prefix']))}")

length of min zip_code value: 5
length of max zip_code value: 5


In [ ]:
# Confirm that the Customers table contains no missing values
customers_clean.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [ ]:
# Confirm that every customer ZIP prefix contains exactly five characters

print(f"min length of zip_code str: {min(customers_clean['customer_zip_code_prefix'].str.len())}")
print(f"max length of zip_code str: {max(customers_clean['customer_zip_code_prefix'].str.len())}")

min length of zip_code str: 5
max length of zip_code str: 5


In [ ]:
# Validate the completeness and uniqueness of the customer_id primary key
print(f"missing customer_id count: {int(customers_clean['customer_id'].isna().sum())}")
print(f"duplicate customer_id count: {int(customers_clean['customer_id'].duplicated().sum())}")
print(f"distinct customer_id count: {customers_clean['customer_id'].nunique()}")

missing customer_id count: 0
duplicate customer_id count: 0
distinct customer_id count: 99441


In [ ]:
# Validate customer_unique_id completeness and identify repeat-customer occurrences
print(f"missing customer_unique_id count: {int(customers_clean['customer_unique_id'].isna().sum())}")
print(f"distinct customer_unique_id count: {customers_clean['customer_unique_id'].nunique()}")
print(f"duplicate customer_unique_id count: {int(customers_clean['customer_unique_id'].duplicated().sum())}")

missing customer_unique_id count: 0
distinct customer_unique_id count: 96096
duplicate customer_unique_id count: 3345


### 3.3 Customer Location Text Standardization

Customer city and state values will be standardized to ensure consistent grouping and filtering. Leading and trailing whitespace will be removed, city names will use lowercase formatting, and state codes will use uppercase formatting.

These transformations preserve the original meaning while reducing the risk of duplicate categories caused by inconsistent text formatting.


In [ ]:
# Standardize customer city values by trimming whitespace and converting to lowercase
customers_clean['customer_city'] = customers_clean['customer_city'].str.strip().str.lower()

In [ ]:
# Standardize customer state values by trimming whitespace and converting to uppercase
customers_clean['customer_state'] = customers_clean['customer_state'].str.strip().str.upper()

In [ ]:
# Validate standardized customer city values
print(f"city blank or whitespace count: {int(((customers_clean['customer_city'].str.isspace()) | (customers_clean['customer_city'] == '')).sum())}")
print(f"city leading space count: {int(customers_clean['customer_city'].str.startswith(' ').sum())}")
print(f"city trailing space count: {int(customers_clean['customer_city'].str.endswith(' ').sum())}")

city blank or whitespace count: 0
city leading space count: 0
city trailing space count: 0


In [ ]:
# Validate standardized customer state values
print(f"state blank or whitespace count: {int(((customers_clean['customer_state'].str.isspace()) | (customers_clean['customer_state'] == '')).sum())}")
print(f"state leading space count: {int(customers_clean['customer_state'].str.startswith(' ').sum())}")
print(f"state trailing space count: {int(customers_clean['customer_state'].str.endswith(' ').sum())}")

state blank or whitespace count: 0
state leading space count: 0
state trailing space count: 0


In [ ]:
# Validate customer state code length, capitalization, and distinct count
print(f"Number of state values whose length is not exactly 2: {int((customers_clean['customer_state'].str.len() != 2).sum())}")
print(f"Number of state values that are not uppercase: {int((~customers_clean['customer_state'].str.isupper()).sum())}")
print(f"Number of distinct states: {customers_clean['customer_state'].nunique()}")

Number of state values whose length is not exactly 2: 0
Number of state values that are not uppercase: 0
Number of distinct states: 27


#### Customer Cleaning Findings

The customer ZIP-code prefixes were converted from integers to five-character strings, restoring leading zeros where required. Customer city values were trimmed, and customer state values were standardized to uppercase two-character codes.

The `customer_id` column remains complete and unique across all 99,441 records. Repeated `customer_unique_id` values were intentionally retained because they represent customers associated with multiple orders and are required for repeat-customer analysis.

No missing, blank, or incorrectly formatted city or state values were identified.

### 3.3 Final Customers Validation

The cleaned Customers table will be validated to confirm that all original records were preserved, no missing values were introduced, the primary key remains unique, and the standardized ZIP-code prefixes contain exactly five characters.


In [ ]:
# Confirm that cleaning preserved the original number of customer records
if customers_raw.shape[0] == customers_clean.shape[0]:
    print("Both customers_raw and customers_clean df's have same rows")
    print(f"customers_raw row: {customers_raw.shape[0]}")
    print(f"customers_clean row: {customers_clean.shape[0]}")
else:
    print("customers_raw and customers_clean df's have different rows")
    print(f"customers_raw row: {customers_raw.shape[0]}")
    print(f"customers_clean row: {customers_clean.shape[0]}")

Both customers_raw and customers_clean df's have same rows
customers_raw row: 99441
customers_clean row: 99441


In [ ]:
customers_clean.shape

(99441, 5)

In [ ]:
customers_clean.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

#### Final Customers Validation Findings

The cleaned Customers table passed all final validation checks:

* All **99,441 customer records** were preserved.
* The table retains its original five columns.
* `customer_id` remains complete and unique.
* No missing values were introduced.
* All ZIP-code prefixes contain exactly five characters.
* Customer city and state values follow the required formatting standards.
* Repeated `customer_unique_id` values were retained for repeat-customer analysis.

### 3.4 Save the Cleaned Customers Table

The validated Customers table will be saved in the processed-data directory without modifying the original raw dataset.


In [ ]:
# Save the validated Customers table to the processed-data directory
customers_clean.to_csv(Processed_Data_Path/'olist_customers_clean.csv', index = False)

## 4. Order Items Cleaning

The Order Items table will be cleaned while preserving its order-item grain. The cleaning process will create an independent working copy, convert `shipping_limit_date` to datetime format, validate the composite key formed by `order_id` and `order_item_id`, and retain valid zero-freight records.

Known future shipping-limit timestamp exceptions will be flagged rather than removed because their correct values cannot be reconstructed reliably.

### 4.1 Create the Clean Order Items Copy

A separate clean copy will be created so that transformations do not modify `order_items_raw`.


In [ ]:
# Create an independent working copy of the raw order_items table
order_items_clean = order_items_raw.copy()
order_items_clean.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [ ]:
if order_items_raw.shape == order_items_clean.shape:
    print("Both order_items_raw and order_items_clean df's have same shapes")
    print(f"order_items_raw shape: {order_items_raw.shape}")
    print(f"order_items_clean shape: {order_items_clean.shape}")
else:
    print("order_items_raw and order_items_clean df's have different shapes")
    print(f"order_items_raw shape: {order_items_raw.shape}")
    print(f"order_items_clean shape: {order_items_clean.shape}")

Both order_items_raw and order_items_clean df's have same shapes
order_items_raw shape: (112650, 7)
order_items_clean shape: (112650, 7)


### 4.2 Shipping Limit Date Conversion

`shipping_limit_date` is currently stored as a string and will be converted to datetime format. Missing-value counts will be compared before and after conversion to confirm that no valid timestamps are lost because of parsing errors.


In [ ]:
#Missing count in shipping_limit_date before conversion.

int(order_items_clean['shipping_limit_date'].isna().sum())

0

In [ ]:
#Convert shipping_limit_date column to datetime format
order_items_clean['shipping_limit_date'] = pd.to_datetime(order_items_clean['shipping_limit_date'], errors='coerce')

In [ ]:
#Missing count after conversion
int(order_items_clean['shipping_limit_date'].isna().sum())

0

In [ ]:
# Confirm that shipping_limit_date uses a datetime data type
order_items_clean.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

#### Shipping Limit Date Conversion Findings

`shipping_limit_date` was successfully converted from string to datetime format. The missing-value count remained zero before and after conversion, confirming that all original timestamp values were parsed successfully.

### 4.3 Future Shipping-Limit Date Exceptions

The data-quality audit identified four order-item records with shipping-limit dates in 2020, although their corresponding orders were placed in 2017. These values will be retained but flagged as timestamp exceptions because their correct shipping-limit dates cannot be reconstructed reliably.


In [ ]:
# Create a flag for shipping-limit dates later than the audited dataset period
flag_future_shipping_limit_date = order_items_clean['shipping_limit_date'] >'2018-12-31'

In [ ]:
# Add and validate the future shipping-limit date exception flag
order_items_clean['flag_future_shipping_limit_date'] = flag_future_shipping_limit_date
int(order_items_clean['flag_future_shipping_limit_date'].sum())

4

### 4.4 Composite-Key and Item-Sequence Validation

Each Order Items record should be uniquely identified by the combination of `order_id` and `order_item_id`. The composite key will be checked for missing and duplicated values.

The `order_item_id` sequence will also be validated to confirm that item numbering begins at one and remains continuous within each order.


In [ ]:
# Validate the completeness and uniqueness of the Order Items composite key
print(f"Missing values in order_id: {int(order_items_clean['order_id'].isna().sum())}")
print(f"Missing values in order_item_id: {int(order_items_clean['order_item_id'].isna().sum())}")
print(f"Duplicated order_id-order_item_id combinations: {int(order_items_clean[['order_id','order_item_id']].duplicated().sum())}")

Missing values in order_id: 0
Missing values in order_item_id: 0
Duplicated order_id-order_item_id combinations: 0


In [ ]:
# Validate the minimum and maximum order_item_id values

print(f"Minimum order_item_id: {min(order_items_clean['order_item_id'])}")
print(f"Maximum order_item_id: {max(order_items_clean['order_item_id'])}")

Minimum order_item_id: 1
Maximum order_item_id: 21


In [ ]:
# Count order_item_id values less than or equal to zero
print(f"number of order_item_id values less than or equal to zero: {int((order_items_clean['order_item_id'] <= 0).sum())}")

number of order_item_id values less than or equal to zero: 0


In [ ]:
# Validate continuous order_item_id sequences within each order
item_count_per_order = order_items_clean.groupby('order_id')['order_item_id'].count()
max_item_id_per_order = order_items_clean.groupby('order_id')['order_item_id'].max()
item_sequence_mismatch = item_count_per_order !=  max_item_id_per_order
int(item_sequence_mismatch.sum())

0

#### Composite-Key and Item-Sequence Findings

The combination of `order_id` and `order_item_id` contains no missing or duplicated values and therefore forms a valid composite key.

`order_item_id` values range from 1 to 21. Each order’s maximum item ID equals its number of item records, confirming that item sequences are continuous within every order.

### 4.5 Price and Freight-Value Validation

The `price` and `freight_value` columns will be validated for missing, negative, and zero values. Valid zero-freight records will be retained because they may represent free-shipping promotions, while invalid negative values would require special treatment.


In [ ]:
# Validate missing, negative, zero, and minimum price values
print(f"Missing values: {int(order_items_clean['price'].isna().sum())}")
print(f"Negative values: {int((order_items_clean['price'] < 0 ).sum())}")
print(f"Zero values: {int((order_items_clean['price'] == 0 ).sum())}")
print(f"Minimum value: {min(order_items_clean['price'] )}")

Missing values: 0
Negative values: 0
Zero values: 0
Minimum value: 0.85


In [ ]:
# Validate missing, negative, zero, and minimum freight values
print(f"Missing values: {int(order_items_clean['freight_value'].isna().sum())}")
print(f"Negative values: {int((order_items_clean['freight_value'] < 0 ).sum())}")
print(f"Zero values: {int((order_items_clean['freight_value'] == 0 ).sum())}")
print(f"Minimum value: {min(order_items_clean['freight_value'] )}")

Missing values: 0
Negative values: 0
Zero values: 383
Minimum value: 0.0


#### Price and Freight-Value Findings

The `price` and `freight_value` columns contain no missing or negative values. All item prices are positive, with a minimum value of 0.85.

A total of 383 records have a freight value of zero. These records will be retained because zero freight can represent valid free-shipping transactions.

### 4.6 Foreign-Key Validation

The Order Items foreign-key columns will be validated against their parent tables to ensure that every referenced order, product, and seller exists. Any unmatched identifiers would prevent reliable joins in downstream analysis.



In [ ]:
# Identify Order Items records whose order_id is absent from Orders
orderid_in_order_items_clean_notin_orders = order_items_clean[~order_items_clean['order_id'].isin(orders_clean['order_id'])]
len(orderid_in_order_items_clean_notin_orders)

0

In [ ]:
# Identify Order Items records whose product_id is absent from Products
productid_in_orderitems_notin_products = order_items_clean[~order_items_clean['product_id'].isin(products_raw['product_id'])]
len(productid_in_orderitems_notin_products)

0

In [ ]:
# Identify Order Items records whose seller_id is absent from Sellers
Sellerid_in_orderitems_notin_sellers = order_items_clean[~order_items_clean['seller_id'].isin(sellers_raw['seller_id'])]
len(Sellerid_in_orderitems_notin_sellers)

0

#### Foreign-Key Validation Findings

Every `order_id`, `product_id`, and `seller_id` in the Order Items table has a matching record in its corresponding parent table. No orphaned Order Items records were identified, so the table can be joined reliably with Orders, Products, and Sellers.

### 4.7 Final Order Items Validation

The cleaned Order Items table will be validated before saving. The validation will confirm that all original records were preserved, no missing values were introduced, the composite key remains unique, the shipping-limit column is datetime, and the future-date exception flag is Boolean.


In [ ]:
#order_items_raw and order_items_clean have the same row count 
if order_items_raw.shape[0] == order_items_clean.shape[0]:
    print("Both order_items_raw and order_items_clean df's have same rows")
    print(f"order_items_raw rows: {order_items_raw.shape[0]}")
    print(f"order_items_clean rows: {order_items_clean.shape[0]}")
else:
    print("order_items_raw and order_items_clean df's have different rows")
    print(f"order_items_raw rows: {order_items_raw.shape[0]}")
    print(f"order_items_clean rows: {order_items_clean.shape[0]}")

Both order_items_raw and order_items_clean df's have same rows
order_items_raw rows: 112650
order_items_clean rows: 112650


In [ ]:
#order_items_clean has 8 columns because one flag was added
print(f"order_iems_clan column count: {order_items_clean.shape[1]}")

order_iems_clan column count: 8


In [ ]:
#The total missing-value count 
order_items_clean.isna().sum()

order_id                           0
order_item_id                      0
product_id                         0
seller_id                          0
shipping_limit_date                0
price                              0
freight_value                      0
flag_future_shipping_limit_date    0
dtype: int64

#### Final Order Items Validation Findings

The cleaned Order Items table passed all final validation checks:

* All **112,650 original records** were preserved.
* The `order_id`–`order_item_id` composite key remains complete and unique.
* Item sequences are continuous within every order.
* `shipping_limit_date` uses a datetime data type.
* No missing or negative price and freight values were identified.
* The 383 valid zero-freight records were retained.
* Four future shipping-limit timestamps were retained and identified through a Boolean exception flag.
* All foreign-key values match their corresponding parent tables.

### 4.8 Save the Cleaned Order Items Table

The validated Order Items table will be saved in the processed-data directory without modifying the original raw dataset.


In [ ]:
# Save the validated order_items table to the processed-data directory
order_items_clean.to_csv(Processed_Data_Path/'olist_order_items_clean.csv', index = False)

## 5. Payments Cleaning

The Payments table will be cleaned while preserving its payment-record grain. The cleaning process will create an independent copy, validate the `order_id`–`payment_sequential` composite key, and flag known payment-sequence, payment-type, installment, and zero-value exceptions.

Payment records will be retained rather than removed because multiple payment records can legitimately belong to the same order.

### 5.1 Create the Clean Payments Copy

A separate clean copy will be created so that transformations and exception flags do not modify `payments_raw`.


In [ ]:
# Create and validate an independent working copy of the raw Payments table
payments_clean = payments_raw.copy()
payments_clean.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [ ]:
if payments_raw.shape == payments_clean.shape:
    print("Both payments_raw and payments_clean df's have same shapes")
    print(f"payments_raw shape: {payments_raw.shape}")
    print(f"payments_clean shape: {payments_clean.shape}")
else:
    print("payments_raw and payments_clean df's have different shapes")
    print(f"payments_raw shape: {payments_raw.shape}")
    print(f"payments_clean shape: {payments_clean.shape}")

Both payments_raw and payments_clean df's have same shapes
payments_raw shape: (103886, 5)
payments_clean shape: (103886, 5)


### 5.2 Missing Values and Composite-Key Validation

The Payments table will be checked for missing values and duplicate `order_id`–`payment_sequential` combinations. This pair represents the payment-record grain because one order may contain multiple payment transactions.


In [ ]:
# Validate missing values and the Payments composite key
payments_clean.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [ ]:
#Duplicated order_id–payment_sequential combinations
int(payments_clean[['order_id', 'payment_sequential']].duplicated().sum())

0

### 5.3 Payment-Sequence Exception Flag

For each order, `payment_sequential` should normally form a continuous sequence beginning with 1. The number of payment records per order will be compared with its maximum payment-sequence value.

Orders with a mismatch will be retained and flagged because the missing sequence records are not available and cannot be reconstructed reliably.


In [ ]:
# Identify orders whose payment sequence is incomplete
payments_count_per_order = payments_clean.groupby('order_id')['payment_sequential'].count()
max_payment_sequence_per_order =  payments_clean.groupby('order_id')['payment_sequential'].max()
payment_sequence_mismatch = payments_count_per_order != max_payment_sequence_per_order
int(payment_sequence_mismatch.sum())

80

In [ ]:
#create mismatched_payment_order_ids
mismatched_payment_order_ids = payment_sequence_mismatch[payment_sequence_mismatch ==True ].index
len(mismatched_payment_order_ids)

80

In [ ]:
#Check whether each payments_clean["order_id"] belongs to mismatched_payment_order_ids.
flag_payment_sequence_mismatch = payments_clean['order_id'].isin(mismatched_payment_order_ids)
flag_payment_sequence_mismatch

0         False
1         False
2         False
3         False
4         False
          ...  
103881    False
103882    False
103883    False
103884    False
103885    False
Name: order_id, Length: 103886, dtype: bool

In [ ]:
# Add and validate the incomplete payment-sequence flag
payments_clean['flag_payment_sequence_mismatch'] = flag_payment_sequence_mismatch
print(f"Flagged payment rows: {payments_clean['flag_payment_sequence_mismatch'].sum()}")
affected_payment_orders = payments_clean[payments_clean['flag_payment_sequence_mismatch']==True]
print(f"Affected orders: {affected_payment_orders['order_id'].nunique()}")

Flagged payment rows: 82
Affected orders: 80


#### Payment-Sequence Findings

Payment-sequence mismatches were identified for **80 orders**, affecting **82 payment records**. These records do not begin with a complete sequence from one, suggesting that earlier payment entries may be absent from the dataset.

The available payment records will be retained and flagged because the missing transactions cannot be reconstructed.

### 5.4 Payment-Value and Attribute Exception Flags

Known exceptions involving undefined payment methods, invalid installment counts, and zero payment values will be retained and identified through Boolean flags. This preserves the original records while allowing them to be handled appropriately during payment analysis.


In [ ]:
# Flag and count records with an undefined payment type
flag_payment_type_not_defined = payments_clean['payment_type']=="not_defined"
payments_clean['flag_payment_type_not_defined'] = flag_payment_type_not_defined
int(payments_clean['flag_payment_type_not_defined'].sum())

3

In [ ]:
# Flag and count records with zero or negative payment installments
flag_invalid_payment_installments = payments_clean['payment_installments'] <= 0
payments_clean['flag_invalid_payment_installments'] = flag_invalid_payment_installments
int(payments_clean['flag_invalid_payment_installments'].sum())

2

In [ ]:
# Flag and count records with a zero payment value
flag_zero_payment_value = payments_clean['payment_value'] == 0
payments_clean['flag_zero_payment_value'] = flag_zero_payment_value
int(payments_clean['flag_zero_payment_value'].sum())

9

In [ ]:
# Create a combined flag for records with any payment-quality exception
flag_payment_quality_exception = (payments_clean['flag_payment_sequence_mismatch']) |  (payments_clean['flag_payment_type_not_defined']) |  (payments_clean['flag_invalid_payment_installments']) |  (payments_clean['flag_zero_payment_value'])
payments_clean['flag_payment_quality_exception'] = flag_payment_quality_exception
int(payments_clean['flag_payment_quality_exception'].sum())

91

#### Payment-Quality Exception Findings

The Payments table contains the following flagged exceptions:

* **82 payment records** belong to 80 orders with incomplete payment sequences.
* **3 records** use the `not_defined` payment type.
* **2 records** have an invalid installment count of zero.
* **9 records** have a payment value of zero.
* The combined payment-quality flag identifies **91 unique payment records**.

The combined count is lower than the sum of the individual counts because some records satisfy multiple exception conditions. All records will be retained and flagged so they can be handled appropriately during payment analysis.

### 5.5 Payments Foreign-Key Validation

Every payment record should reference an existing order. The `order_id` foreign key will be validated against the cleaned Orders table before the Payments table is saved.


In [ ]:
# Identify Payment records whose order_id is absent from Orders
orderid_in_payments_notin_orders = payments_clean[~payments_clean['order_id'].isin(orders_clean['order_id'])]
len(orderid_in_payments_notin_orders)

0

#### Payments Foreign-Key Findings

All `order_id` values in the Payments table have a matching record in the cleaned Orders table. No orphaned payment records were identified.

### 5.6 Final Payments Validation

The cleaned Payments table will be validated before saving. The validation will confirm that all original records were preserved, no missing values were introduced, the composite key remains unique, and all payment-quality flags use the Boolean data type.


In [ ]:
#payments_raw and payments_clean have the same row count 
if payments_raw.shape[0] == payments_clean.shape[0]:
    print("Both payments_raw and payments_clean df's have same rows")
    print(f"payments_raw rows: {payments_raw.shape[0]}")
    print(f"payments_clean rows: {payments_clean.shape[0]}")
else:
    print("payments_raw and payments_clean df's have different rows")
    print(f"payments_raw rows: {payments_raw.shape[0]}")
    print(f"payments_clean rows: {payments_clean.shape[0]}")

Both payments_raw and payments_clean df's have same rows
payments_raw rows: 103886
payments_clean rows: 103886


In [ ]:
#number of columns in payments_clean
len(payments_clean.columns)

10

In [ ]:
#Verify missing values
payments_clean.isna().sum()

order_id                             0
payment_sequential                   0
payment_type                         0
payment_installments                 0
payment_value                        0
flag_payment_sequence_mismatch       0
flag_payment_type_not_defined        0
flag_invalid_payment_installments    0
flag_zero_payment_value              0
flag_payment_quality_exception       0
dtype: int64

In [ ]:
# Data types of flag columns 
flag_columns = ['flag_payment_sequence_mismatch', 'flag_payment_type_not_defined', 'flag_invalid_payment_installments', 'flag_zero_payment_value', 'flag_payment_quality_exception']
payments_clean[flag_columns].dtypes

flag_payment_sequence_mismatch       bool
flag_payment_type_not_defined        bool
flag_invalid_payment_installments    bool
flag_zero_payment_value              bool
flag_payment_quality_exception       bool
dtype: object

#### Final Payments Validation Findings

The cleaned Payments table passed all final validation checks:

* All **103,886 original payment records** were preserved.
* The `order_id`–`payment_sequential` composite key remains complete and unique.
* No missing values were introduced.
* All payment records reference an existing order.
* Five payment-quality flags were added and stored as Boolean columns.
* A total of **91 payment records** contain at least one documented payment-quality exception.

### 5.7 Save the Cleaned Payments Table

The validated Payments table will be saved in the processed-data directory without modifying the original raw dataset.


In [ ]:
# Save the validated Payments table to the processed-data directory
payments_clean.to_csv(Processed_Data_Path/'olist_order_payments_clean.csv', index = False)

## 6. Reviews Cleaning

The Reviews table will be cleaned while preserving all review records. The cleaning process will create an independent copy, convert the review timestamps to datetime format, retain optional missing comment text, and flag unusually long response durations.

Review identifiers will not be treated as standalone primary keys because the audit found repeated `review_id` and `order_id` values. Their relationship will be preserved for order-level review analysis.

### 6.1 Create the Clean Reviews Copy

A separate clean copy will be created so that all transformations and exception flags can be applied without modifying `reviews_raw`.


In [ ]:
# Create and validate an independent working copy of the raw Reviews table
reviews_clean = reviews_raw.copy()
reviews_clean.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [ ]:
if reviews_raw.shape == reviews_clean.shape:
    print("Both reviews_raw and reviews_clean df's have same shapes")
    print(f"reviews_raw shape: {reviews_raw.shape}")
    print(f"reviews_clean shape: {reviews_clean.shape}")
else:
    print("reviews_raw and reviews_clean df's have different shapes")
    print(f"reviews_raw shape: {reviews_raw.shape}")
    print(f"reviews_clean shape: {reviews_clean.shape}")

Both reviews_raw and reviews_clean df's have same shapes
reviews_raw shape: (99224, 7)
reviews_clean shape: (99224, 7)


### 6.2 Review Timestamp Conversion

`review_creation_date` and `review_answer_timestamp` are currently stored as strings. Both columns will be converted to datetime format, and their missing-value counts will be compared before and after conversion to detect parsing failures.


In [ ]:
#Store the two timestamp column names in a list.
reviews_time_stamp_cols = ['review_creation_date', 'review_answer_timestamp']


In [ ]:
# Missing counts before timestamp conversion
reviews_clean[reviews_time_stamp_cols].isna().sum()

review_creation_date       0
review_answer_timestamp    0
dtype: int64

In [ ]:
# Convert review timestamp columns to datetime
for cols in reviews_time_stamp_cols: 
    reviews_clean[cols] = pd.to_datetime(reviews_clean[cols], errors='coerce')

In [ ]:
# Missing counts after timestamp conversion
reviews_clean[reviews_time_stamp_cols].isna().sum()

review_creation_date       0
review_answer_timestamp    0
dtype: int64

In [ ]:
# Confirm that both review timestamp columns use a datetime data type
reviews_clean.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

#### Review Timestamp Conversion Findings

Both review timestamp columns were successfully converted from string to datetime format. Their missing-value counts remained zero before and after conversion, confirming that no parsing failures were introduced.

### 6.3 Review Timestamp Sequence and Duration Validation

The review timestamps will be checked to confirm that no answer timestamp occurs before its review-creation date. The response duration will then be calculated, and records with durations greater than 30 days will be flagged as unusually long rather than removed.


In [ ]:
# Count reviews answered before their creation date
len(reviews_clean[reviews_clean['review_answer_timestamp'] < reviews_clean['review_creation_date']])

0

In [ ]:
# Calculate the duration between review creation and answer timestamps
review_response_duration = reviews_clean['review_answer_timestamp'] - reviews_clean['review_creation_date'] 
reviews_clean['review_response_duration'] = review_response_duration
reviews_clean['review_response_duration'].describe()

count                     99224
mean     3 days 03:34:33.029700
std      9 days 21:21:40.258026
min             0 days 02:08:29
25%      1 days 00:07:00.750000
50%      1 days 16:11:55.500000
75%             3 days 02:29:08
max           518 days 16:46:52
Name: review_response_duration, dtype: object

In [ ]:
#Boolean flag for records where review_response_duration is strictly greater than 30 days.
flag_review_response_over_30_days = reviews_clean['review_response_duration'] > pd.Timedelta(days=30)
int(flag_review_response_over_30_days.sum())

682

In [ ]:
# Add and validate the long review-response duration flag
reviews_clean['flag_review_response_over_30_days'] = flag_review_response_over_30_days
reviews_clean['flag_review_response_over_30_days'].dtypes

dtype('bool')

#### Review Timestamp Findings

No review-answer timestamps occur before their corresponding review-creation dates. Review response duration ranges from approximately two hours to more than 518 days.

A total of **682 records** have response durations greater than 30 days. These records were retained and identified using a Boolean exception flag.

### 6.4 Optional Review-Text Treatment

Missing review titles and messages are expected because customers can submit a score without written feedback. These values will remain missing rather than being replaced with artificial text.

Existing non-missing titles and messages will be trimmed to remove surrounding whitespace. Any values that become empty after trimming will be treated as missing.


In [ ]:
# Remove leading and trailing whitespace from review titles
reviews_clean['review_comment_title'] = reviews_clean['review_comment_title'].str.strip()

In [ ]:
# Remove leading and trailing whitespace from review messages
reviews_clean['review_comment_message'] = reviews_clean['review_comment_message'].str.strip()

In [ ]:
# Count review titles that are empty after trimming
empty_reviews_title = reviews_clean[reviews_clean['review_comment_title'] == ""]
len(empty_reviews_title)

2

In [ ]:
# Count review messages that are empty after trimming
empty_reviews_messages = reviews_clean[reviews_clean['review_comment_message'] == ""]
len(empty_reviews_messages)

27

In [ ]:
# Convert empty review titles to missing values
reviews_clean['review_comment_title'] = reviews_clean['review_comment_title'].replace("", pd.NA)

# Count missing review titles after standardization
int(reviews_clean['review_comment_title'].isna().sum())

87658

In [ ]:
# Convert empty review messages to missing values
reviews_clean['review_comment_message'] = reviews_clean['review_comment_message'].replace("", pd.NA)

# Count missing review messages after standardization
int(reviews_clean['review_comment_message'].isna().sum())

58274

#### Optional Review-Text Findings

After trimming whitespace, 2 empty review titles and 27 empty review messages were identified and standardized as missing values.

The cleaned table contains **87,658 missing titles** and **58,274 missing messages**. These values will remain missing because written comments are optional and cannot be reconstructed.

### 6.5 Review Score and Identifier Validation

Review scores will be validated against the expected range of 1 through 5. Review identifiers and order identifiers will also be checked while preserving their known many-to-many-like behavior within this source table.


In [ ]:
# Validate review-score completeness and allowed range
print(f"Missing values: {int(reviews_clean['review_score'].isna().sum())}")
print(f"Minimum score: {min(reviews_clean['review_score'])}")
print(f"Maximum score: {max(reviews_clean['review_score'])}")
print(f"Values below 1 or above 5: {len(reviews_clean[(reviews_clean['review_score']<1) | (reviews_clean['review_score']>5)])}")

Missing values: 0
Minimum score: 1
Maximum score: 5
Values below 1 or above 5: 0


In [ ]:
# Validate review identifiers and duplicate patterns
print(f"Missing review_id values: {int(reviews_clean['review_id'].isna().sum())}")
print(f"Missing order_id values: {int(reviews_clean['order_id'].isna().sum())}")
print(f"Exact duplicate rows: {int(reviews_clean.duplicated().sum())}")
print(f"Duplicated review_id-order_id combinations: {int(reviews_clean[['review_id', 'order_id']].duplicated().sum())}")
print(f"Duplicated review_id occurrences: {int(reviews_clean['review_id'].duplicated().sum())}")
print(f"Duplicated order_id occurrences: {int(reviews_clean['order_id'].duplicated().sum())}")


Missing review_id values: 0


Missing order_id values: 0
Exact duplicate rows: 0
Duplicated review_id-order_id combinations: 0
Duplicated review_id occurrences: 814
Duplicated order_id occurrences: 551


#### Review Score and Identifier Findings

All review scores are complete and fall within the valid range of 1 through 5. No exact duplicate rows or duplicated `review_id`–`order_id` combinations were identified.

The table contains 814 duplicated `review_id` occurrences and 551 duplicated `order_id` occurrences. These records were retained because neither identifier is a valid standalone primary key in this source table, while each `review_id`–`order_id` pair remains unique.

### 6.6 Reviews Foreign-Key Validation

Every review should reference an existing order. The `order_id` foreign key will be validated against the cleaned Orders table before the Reviews table is saved.


In [ ]:
# Identify Review records whose order_id is absent from Orders
orderid_in_reviews_notin_orders = reviews_clean[~reviews_clean['order_id'].isin(orders_clean['order_id'])]
len(orderid_in_reviews_notin_orders)

0

#### Reviews Foreign-Key Findings

All `order_id` values in the Reviews table have a matching record in the cleaned Orders table. No orphaned review records were identified.

### 6.7 Final Reviews Validation

The cleaned Reviews table will be validated before saving. The validation will confirm that all original records were preserved, only optional review-text fields contain missing values, timestamp and duration columns use appropriate data types, and the long-duration exception flag is Boolean.


In [ ]:
#reviews_raw and reviews_clean have the same row count 
if reviews_raw.shape[0] == reviews_clean.shape[0]:
    print("Both reviews_raw and reviews_clean df's have same rows")
    print(f"reviews_raw rows: {reviews_raw.shape[0]}")
    print(f"reviews_clean rows: {reviews_clean.shape[0]}")
else:
    print("reviews_raw and reviews_clean df's have different rows")
    print(f"reviews_raw rows: {reviews_raw.shape[0]}")
    print(f"reviews_clean rows: {reviews_clean.shape[0]}")

Both reviews_raw and reviews_clean df's have same rows
reviews_raw rows: 99224
reviews_clean rows: 99224


In [ ]:
len(reviews_clean.columns)

9

In [ ]:
reviews_clean.isna().sum()

review_id                                0
order_id                                 0
review_score                             0
review_comment_title                 87658
review_comment_message               58274
review_creation_date                     0
review_answer_timestamp                  0
review_response_duration                 0
flag_review_response_over_30_days        0
dtype: int64

In [ ]:
reviews_clean.dtypes

review_id                                        str
order_id                                         str
review_score                                   int64
review_comment_title                             str
review_comment_message                           str
review_creation_date                  datetime64[us]
review_answer_timestamp               datetime64[us]
review_response_duration             timedelta64[us]
flag_review_response_over_30_days               bool
dtype: object

#### Final Reviews Validation Findings

The cleaned Reviews table passed all final validation checks:

* All **99,224 original review records** were preserved.
* Both review timestamp columns use datetime data types.
* `review_response_duration` uses a timedelta data type.
* The long-duration exception flag uses the Boolean data type.
* Review scores remain complete and within the valid range of 1 through 5.
* Missing values occur only in the optional review-title and review-message columns.
* Empty written-feedback values were standardized as missing.
* All review records reference an existing order.

### 6.8 Save the Cleaned Reviews Table

The validated Reviews table will be saved in the processed-data directory without modifying the original raw dataset.


In [ ]:
# Save the validated Reviews table to the processed-data directory
reviews_clean.to_csv(Processed_Data_Path/'olist_order_reviews_clean.csv', index = False)


## 7. Products and Category Translation Cleaning

The Products and Category Translation tables will be cleaned together. All product records will be preserved while missing product attributes, missing dimensions, zero weights, and untranslated categories are identified through exception flags.

The misspelled source column names containing `lenght` will be corrected to `length` in the cleaned Products table. The raw tables will remain unchanged.

### 7.1 Create the Clean Products and Translation Copies

Separate working copies will be created so that all transformations can be performed without modifying `products_raw` or `translation_raw`.


In [ ]:
#Create products_clean as a copy of products_raw.
#Create translation_clean as a copy of translation_raw.

products_clean = products_raw.copy()
translation_clean = translation_raw.copy()


In [ ]:
#Products - Compare their shapes with their respective raw tables.
if products_raw.shape == products_clean.shape:
    print("Both products_raw and products_clean df's have same shapes")
    print(f"products_raw shape: {products_raw.shape}")
    print(f"products_clean shape: {products_clean.shape}")
else:
    print("products_raw and products_clean df's have different shapes")
    print(f"products_raw shape: {products_raw.shape}")
    print(f"products_clean shape: {products_clean.shape}")


Both products_raw and products_clean df's have same shapes
products_raw shape: (32951, 9)
products_clean shape: (32951, 9)


In [ ]:
#Translation - Compare their shapes with their respective raw tables.
if translation_raw.shape == translation_clean.shape:
    print("Both translation_raw and translation_clean df's have same shapes")
    print(f"translation_raw shape: {translation_raw.shape}")
    print(f"translation_clean shape: {translation_clean.shape}")
else:
    print("translation_raw and translation_clean df's have different shapes")
    print(f"translation_raw shape: {translation_raw.shape}")
    print(f"translation_clean shape: {translation_clean.shape}")


Both translation_raw and translation_clean df's have same shapes
translation_raw shape: (71, 2)
translation_clean shape: (71, 2)


### 7.2 Product Column-Name Standardization

The source Products table contains two misspelled column names using `lenght`. These names will be corrected in `products_clean` to improve clarity and prevent the misspellings from propagating into downstream analysis.


In [ ]:
# Correct misspelled product length column names
products_clean = products_clean.rename(columns={"product_name_lenght":"product_name_length", "product_description_lenght":"product_description_length"})
products_clean.columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

### 7.3 Missing Product Attribute Flags

Missing product attributes will not be imputed because their correct category, description, photo quantity, weight, and dimensions cannot be determined reliably.

Two Boolean flags will be created:

* A catalog-attribute flag for products missing category, name length, description length, or photo quantity.
* A physical-attribute flag for products missing weight, length, height, or width.

These products will remain available for sales analysis but can be excluded from analyses requiring the missing attributes.


In [ ]:
# Create a flag for products missing any catalog attribute

flag_missing_catalog_attributes = (products_clean['product_category_name'].isna()) | (products_clean['product_name_length'].isna()) | (products_clean['product_description_length'].isna()) | (products_clean['product_photos_qty'].isna())
products_clean['flag_missing_catalog_attributes'] = flag_missing_catalog_attributes
int(products_clean['flag_missing_catalog_attributes'].sum())

610

In [ ]:
# Create and validate a flag for products missing any physical attribute
flag_missing_physical_attributes = (products_clean['product_weight_g'].isna()) | (products_clean['product_length_cm'].isna()) | (products_clean['product_height_cm'].isna()) | (products_clean['product_width_cm'].isna())
products_clean['flag_missing_physical_attributes'] = flag_missing_physical_attributes
int(products_clean['flag_missing_physical_attributes'].sum())

2

In [ ]:
# Create and validate a flag for products with zero recorded weight
flag_zero_product_weight = products_clean['product_weight_g'] == 0
products_clean['flag_zero_product_weight'] = flag_zero_product_weight
int(products_clean['flag_zero_product_weight'].sum())

4

#### Missing Product Attribute Findings

The Products table contains **610 products** missing at least one catalog attribute and **2 products** missing at least one physical attribute. Four additional products have a recorded weight of zero.

All affected products were retained and identified through Boolean flags. They remain usable for sales analysis but should be excluded from analyses requiring their missing or invalid attributes.

### 7.4 Category Translation Preparation

Product category values will be standardized before connecting Products with the Category Translation table. Surrounding whitespace will be removed, and category names will use lowercase formatting to support reliable matching.


In [ ]:
# Standardize Portuguese and English product category names

products_clean['product_category_name'] = products_clean['product_category_name'].str.strip().str.lower()
translation_clean['product_category_name'] = translation_clean['product_category_name'].str.strip().str.lower()
translation_clean['product_category_name_english'] = translation_clean['product_category_name_english'].str.strip().str.lower()

In [ ]:
# Validate the cleaned Category Translation table
print(f"Missing values in both column : {translation_clean.isna().sum()}")
print(f"blank values in product_category_name: {int((translation_clean['product_category_name'] == '').sum())}")
print(f"blank values in product_category_name_english: {int((translation_clean['product_category_name_english'] == '').sum())}")
print(f"duplicated product_category_name: {translation_clean['product_category_name'].duplicated().sum()}")
print(f"distinct product_category_name: {translation_clean['product_category_name'].nunique()}")

Missing values in both column : product_category_name            0
product_category_name_english    0
dtype: int64
blank values in product_category_name: 0
blank values in product_category_name_english: 0
duplicated product_category_name: 0
distinct product_category_name: 71


In [ ]:
# Create and validate a flag for non-missing product categories absent from the translation table

flag_untranslated_category =  products_clean['product_category_name'].notna() & ~products_clean['product_category_name'].isin(translation_clean['product_category_name'])
products_clean['flag_untranslated_category'] = flag_untranslated_category
int(products_clean['flag_untranslated_category'].sum())

13

In [ ]:
# Inspect the distinct untranslated product categories and their frequencies
products_flag_untranslated_category_true = products_clean[products_clean['flag_untranslated_category'] == True]
products_flag_untranslated_category_true['product_category_name'].nunique()

2

In [ ]:
products_flag_untranslated_category_true['product_category_name'].value_counts()

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64

#### Category Translation Findings

The cleaned translation table contains 71 complete and unique Portuguese-to-English category mappings.

Thirteen product records belong to two non-missing categories that are absent from the translation table: `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`. These products were retained and flagged.

### 7.5 Add English Category Names

English category names will be added to `products_clean` using a left join with `translation_clean`. A left join is required to preserve all 32,951 product records, including products with missing or untranslated categories.


In [ ]:
# Add English category names while preserving all product records
products_clean=pd.merge(products_clean, translation_clean, on = 'product_category_name', how = 'left')

In [ ]:
products_clean

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,flag_missing_catalog_attributes,flag_missing_physical_attributes,flag_zero_product_weight,flag_untranslated_category,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,False,False,False,False,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,False,False,False,False,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,False,False,False,False,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,False,False,False,False,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,False,False,False,False,housewares
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0,False,False,False,False,furniture_decor
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0,False,False,False,False,construction_tools_lights
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0,False,False,False,False,bed_bath_table
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0,False,False,False,False,computers_accessories


In [ ]:
products_clean.shape

(32951, 14)

In [ ]:
#Missing product_category_name_english count
products_clean['product_category_name_english'].isna().sum()

np.int64(623)

In [ ]:
#create an analysis-ready category column named:product_category_name_analysis

product_category_name_analysis = products_clean['product_category_name_english'].isin(translation_clean['product_category_name_english'])


In [ ]:
# Create an analysis-ready category using English, Portuguese, then unknown
products_clean["product_category_name_analysis"] = (products_clean["product_category_name_english"].fillna(products_clean["product_category_name"]).fillna("unknown"))

In [ ]:
# Confirm that the analysis-ready category contains no missing values
products_clean["product_category_name_analysis"].isna().sum()

np.int64(0)

In [ ]:
# Count products assigned to the unknown analysis category
int((products_clean['product_category_name_analysis'] == "unknown").sum())

610

In [ ]:
# Validate unknown and untranslated category fallback values
flag_untranslated_category_true = products_clean[products_clean['flag_untranslated_category'] == True]
flag_untranslated_category_true['product_category_name_analysis'].value_counts()

product_category_name_analysis
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64

#### English Category Integration Findings

The left join preserved all 32,951 product records and added English category names where translations were available.

The analysis-ready category column uses English names when available, retains Portuguese names for the 13 untranslated products, and assigns `unknown` to the 610 products with no original category. It contains no missing values.

### 7.6 Product Key and Numeric Validation

The `product_id` primary key and numeric product attributes will be validated before saving. Missing and zero-value conditions already identified through flags will be retained, while negative measurements would be treated as invalid.


In [ ]:
# Validate the completeness and uniqueness of the product_id primary key

print(f"Missing count : {int(products_clean['product_id'].isna().sum())}")
print(f"Duplicated count : {int(products_clean['product_id'].duplicated().sum())}")
print(f"Distinct count : {products_clean['product_id'].nunique()}")

Missing count : 0
Duplicated count : 0
Distinct count : 32951


In [ ]:
# Validate negative and zero values across numeric product attributes
print(f"Number of negative values - product_name_length: {int((products_clean['product_name_length'] < 0).sum())}")
print(f"Number of negative values - product_description_length: {int((products_clean['product_description_length'] < 0).sum())}")
print(f"Number of negative values - product_photos_qty: {int((products_clean['product_photos_qty'] < 0).sum())}")
print(f"Number of negative values - product_weight_g: {int((products_clean['product_weight_g'] < 0).sum())}")
print(f"Number of negative values - product_length_cm: {int((products_clean['product_length_cm'] < 0).sum())}")
print(f"Number of negative values - product_height_cm: {int((products_clean['product_height_cm'] < 0).sum())}")
print(f"Number of negative values - product_width_cm: {int((products_clean['product_width_cm'] < 0).sum())}")

print(f"Number of zero values - product_name_length: {int((products_clean['product_name_length'] == 0).sum())}")
print(f"Number of zero values - product_description_length: {int((products_clean['product_description_length'] == 0).sum())}")
print(f"Number of zero values - product_photos_qty: {int((products_clean['product_photos_qty'] == 0).sum())}")
print(f"Number of zero values - product_weight_g: {int((products_clean['product_weight_g'] == 0).sum())}")
print(f"Number of zero values - product_length_cm: {int((products_clean['product_length_cm'] == 0).sum())}")
print(f"Number of zero values - product_height_cm: {int((products_clean['product_height_cm'] == 0).sum())}")
print(f"Number of zero values - product_width_cm: {int((products_clean['product_width_cm'] == 0).sum())}")

Number of negative values - product_name_length: 0
Number of negative values - product_description_length: 0
Number of negative values - product_photos_qty: 0
Number of negative values - product_weight_g: 0
Number of negative values - product_length_cm: 0
Number of negative values - product_height_cm: 0
Number of negative values - product_width_cm: 0
Number of zero values - product_name_length: 0
Number of zero values - product_description_length: 0
Number of zero values - product_photos_qty: 0
Number of zero values - product_weight_g: 4
Number of zero values - product_length_cm: 0
Number of zero values - product_height_cm: 0
Number of zero values - product_width_cm: 0


#### Product Numeric-Value Findings

No negative values were identified in any numeric product attribute. Zero values occur only in `product_weight_g`, affecting the four products already identified by the zero-weight exception flag.

No numeric values were removed or imputed because the actual missing or corrected measurements cannot be reconstructed reliably.

### 7.6 Final Products and Translation Validation

The cleaned Products and Category Translation tables will be validated before saving. The validation will confirm that all source records were preserved, the product primary key remains unique, exception flags are Boolean, and the analysis-ready category column is complete.


In [ ]:
#validate for products_clean
print(f"products_clean row count: {products_clean.shape[0]}")
print(f"products_clean column count: {products_clean.shape[1]}")
print(f"product_category_name_english missing count: {int(products_clean['product_category_name_english'].isna().sum())}")
print(f"product_category_name_analysis missing count: {int(products_clean['product_category_name_analysis'].isna().sum())}")


products_clean row count: 32951
products_clean column count: 15
product_category_name_english missing count: 623
product_category_name_analysis missing count: 0


In [ ]:
#missing values in original Attributes

numeric_product_attributes = ['product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
products_clean[numeric_product_attributes].isna().sum()

product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [ ]:
#flag data type validation
products_clean_flags = ['flag_missing_catalog_attributes', 'flag_missing_physical_attributes', 'flag_zero_product_weight', 'flag_untranslated_category']
products_clean[products_clean_flags].dtypes

flag_missing_catalog_attributes     bool
flag_missing_physical_attributes    bool
flag_zero_product_weight            bool
flag_untranslated_category          bool
dtype: object

In [ ]:
#validate translation_clean
print(f"translation_clean shape: {translation_clean.shape}")
print(f"translation_clean missing value count: {translation_clean.isna().sum()}")
print(f"translation_clean duplicated: {int(translation_clean['product_category_name'].duplicated().sum())}")

translation_clean shape: (71, 2)
translation_clean missing value count: product_category_name            0
product_category_name_english    0
dtype: int64
translation_clean duplicated: 0


#### Final Products and Translation Validation Findings

* All **32,951 product records** were preserved during cleaning.
* The `product_id` primary key remains complete and unique.
* Misspelled product-length column names were corrected.
* No negative values were found in the numeric product attributes.
* **610 products** have missing catalog attributes, **2 products** have missing physical attributes, and **4 products** have zero recorded weight. These records were retained and flagged.
* The English category column contains **623 missing values**: 610 products have no original category and 13 products belong to two categories absent from the translation table.
* The analysis-ready category column contains no missing values because it uses the English category where available, the original Portuguese category as a fallback, and `unknown` when the original category is missing.
* The cleaned translation table contains **71 complete and unique category mappings**.
* All product-quality flag columns use the Boolean data type.

### 7.7 Save the Cleaned Products and Translation Tables

The validated Products and Category Translation tables will be exported to the processed-data directory without modifying the original raw datasets.


In [ ]:
# Save the validated Products table to the processed-data directory
products_clean.to_csv(Processed_Data_Path/'olist_products_clean.csv', index = False)

In [ ]:
# Save the validated Translation table to the processed-data directory
translation_clean.to_csv(Processed_Data_Path/'product_category_name_translation_clean.csv', index = False)

## 8. Sellers Table Cleaning

This section prepares the Sellers table for analysis by standardizing ZIP-code prefixes, city names, and state codes while retaining the original records. Previously identified seller-location inconsistencies will be preserved and flagged rather than removed.

### 8.1 Create the Clean Sellers Copy

In [ ]:
#Create sellers_clean as a copy of sellers_raw.
sellers_clean = sellers_raw.copy()

In [ ]:
#Sellers - Compare their shapes with their respective raw tables.
if sellers_raw.shape == sellers_clean.shape:
    print("Both sellers_raw and sellers_clean df's have same shapes")
    print(f"sellers_raw shape: {sellers_raw.shape}")
    print(f"sellers_clean shape: {sellers_clean.shape}")
else:
    print("sellers_raw and sellers_clean df's have different shapes")
    print(f"sellers_raw shape: {sellers_raw.shape}")
    print(f"sellers_clean shape: {sellers_clean.shape}")


Both sellers_raw and sellers_clean df's have same shapes
sellers_raw shape: (3095, 4)
sellers_clean shape: (3095, 4)


### 8.2 Seller ZIP-Code Standardization

The `seller_zip_code_prefix` column will be converted from an integer to a five-character string. Leading zeros will be added where required so that all ZIP-code prefixes follow a consistent format.


In [ ]:
# Convert seller ZIP-code prefixes to five-character strings with leading zeros
sellers_clean['seller_zip_code_prefix'] = sellers_clean['seller_zip_code_prefix'].astype(str)
sellers_clean['seller_zip_code_prefix'] = sellers_clean['seller_zip_code_prefix'].str.zfill(5)

In [ ]:
# Confirm that every seller ZIP prefix contains exactly five characters

print(f"min length of zip_code str: {min(sellers_clean['seller_zip_code_prefix'].str.len())}")
print(f"max length of zip_code str: {max(sellers_clean['seller_zip_code_prefix'].str.len())}")

min length of zip_code str: 5
max length of zip_code str: 5


### 8.3 Seller City and State Standardization

Seller city values will be standardized by removing surrounding whitespace and converting the text to lowercase. Seller state codes will be trimmed and converted to uppercase to maintain a consistent two-character format.


In [ ]:
# Standardize seller city values by trimming whitespace and converting to lowercase
sellers_clean['seller_city'] = sellers_clean['seller_city'].str.strip().str.lower()

In [ ]:
# Standardize seller state codes by trimming whitespace and converting to uppercase
sellers_clean['seller_state'] = sellers_clean['seller_state'].str.strip().str.upper()

In [ ]:
#Missing values in all Sellers columns.
sellers_clean.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [ ]:
#Blank city and state values.
print(f"blank city values: {int((sellers_clean['seller_city'] == '').sum())}")
print(f"blank state values: {int((sellers_clean['seller_state'] == '').sum())}")


blank city values: 0
blank state values: 0


In [ ]:
# Count seller city values containing leading or trailing whitespace

sellers_city_leading_or_trailing_space = sellers_clean['seller_city'].str.startswith(' ') | sellers_clean['seller_city'].str.endswith(' ')
print(f"city value having leading or trailing space count: {int(sellers_city_leading_or_trailing_space.sum())}")

# Count seller state values containing leading or trailing whitespace
sellers_state_leading_or_trailing_space = sellers_clean['seller_state'].str.startswith(' ') | sellers_clean['seller_state'].str.endswith(' ')
print(f"state value having leading or trailing space count: {int(sellers_state_leading_or_trailing_space.sum())}")


city value having leading or trailing space count: 0
state value having leading or trailing space count: 0


In [ ]:
#State codes that are not exactly two uppercase characters
seller_state_not2_notupper = ((sellers_clean['seller_state'].str.len() !=2) | (~sellers_clean['seller_state'].str.isupper()))
int(seller_state_not2_notupper.sum())

0

In [ ]:
#The number of distinct seller states.
sellers_clean['seller_state'].nunique()

23

#### Seller Location Standardization Findings

The seller location fields were successfully standardized. No blank city or state values were found, and no values contain leading or trailing whitespace. All seller-state values consist of exactly two uppercase characters. The Sellers table contains 23 distinct state codes.

### 8.4 Seller Primary-Key Validation

The `seller_id` column will be checked for missing values and duplicates to confirm that it uniquely identifies each seller record.


In [ ]:
# Validate the completeness and uniqueness of the seller_id primary key

print(f"Missing seller_id values : {int(sellers_clean['seller_id'].isna().sum())}")
print(f"Duplicated seller_id values : {int(sellers_clean['seller_id'].duplicated().sum())}")
print(f"Distinct seller_id count : {sellers_clean['seller_id'].nunique()}")
print(f"Total number of seller rows: {sellers_clean.shape[0]}")

Missing seller_id values : 0
Duplicated seller_id values : 0
Distinct seller_id count : 3095
Total number of seller rows: 3095


In [ ]:
sellers_clean['seller_id'].nunique()

3095

In [ ]:
# Identify Order Items records whose seller_id is absent from Sellers
sellerid_in_orderitems_notin_sellers = order_items_clean[~order_items_clean['seller_id'].isin(sellers_clean['seller_id'])]
len(sellerid_in_orderitems_notin_sellers)

0

#### Seller Key and Referential-Integrity Findings

The `seller_id` column is complete and unique: all 3,095 seller records have distinct identifiers, with no missing or duplicated values. Additionally, every `seller_id` referenced in the cleaned Order Items table exists in the cleaned Sellers table. Therefore, the seller key relationship is complete and suitable for joining the two datasets.

### 8.5 Seller Location-Consistency Flags

The data-quality audit identified cases where a seller’s recorded state did not agree with the geographic reference state associated with its ZIP-code prefix. These records will be retained and flagged so that they can be excluded or reviewed when performing location-based seller analysis.


In [ ]:
#Create geolocation_clean as a copy of geolocation_raw
geolocation_clean = geolocation_raw.copy()

In [ ]:
# Confirm that the clean Geolocation copy initially matches the raw table
if geolocation_raw.shape == geolocation_clean.shape:
    print("Both geolocation_raw and geolocation_clean df's have same shapes")
    print(f"geolocation_raw shape: {geolocation_raw.shape}")
    print(f"geolocation_clean shape: {geolocation_clean.shape}")
else:
    print("geolocation_raw and geolocation_clean df's have different shapes")
    print(f"geolocation_raw shape: {geolocation_raw.shape}")
    print(f"geolocation_clean shape: {geolocation_clean.shape}")


Both geolocation_raw and geolocation_clean df's have same shapes
geolocation_raw shape: (1000163, 5)
geolocation_clean shape: (1000163, 5)


#### 8.5.1 Prepare the Geolocation Reference

A working copy of the Geolocation table will be standardized and used to determine the reference state for each seller ZIP-code prefix. This allows seller-state inconsistencies to be identified without changing or removing the original seller records.


In [ ]:
# Convert Geolocation ZIP-code prefixes to five-character strings with leading zeros
geolocation_clean['geolocation_zip_code_prefix'] = geolocation_clean['geolocation_zip_code_prefix'].astype(str)
geolocation_clean['geolocation_zip_code_prefix'] = geolocation_clean['geolocation_zip_code_prefix'].str.zfill(5)

In [ ]:
# Confirm that every geolocation ZIP prefix contains exactly five characters

print(f"min length of zip_code str: {min(geolocation_clean['geolocation_zip_code_prefix'].str.len())}")
print(f"max length of zip_code str: {max(geolocation_clean['geolocation_zip_code_prefix'].str.len())}")

min length of zip_code str: 5
max length of zip_code str: 5


In [ ]:
#standardize geolocation_state 
geolocation_clean['geolocation_state'] = geolocation_clean['geolocation_state'].str.strip().str.upper()

In [ ]:
#Validating that no state value is blank
int((geolocation_clean['geolocation_state'] == '').sum())

0

In [ ]:
#Validating that every state code contains exactly two uppercase characters.
int(((geolocation_clean['geolocation_state'].str.len() != 2) | (~geolocation_clean['geolocation_state'].str.isupper())).sum())

0

In [ ]:
#Count the exact duplicate rows
geolocation_clean.duplicated().sum()

np.int64(261831)

In [ ]:
#Record the current row count.
len(geolocation_clean)

1000163

In [ ]:
# Remove exact duplicate Geolocation records and reset the index
geolocation_clean = geolocation_clean.drop_duplicates()
geolocation_clean = geolocation_clean.reset_index(drop=True)

In [ ]:
#Confirm the new row count
len(geolocation_clean)

738332

In [ ]:
#Confirm the remaining exact-duplicate count
geolocation_clean.duplicated().sum()

np.int64(0)

In [ ]:
# Validate the shape, columns, and duplicate count after Geolocation deduplication
print(f"geolocation shape: {geolocation_clean.shape}")
print(f"exact-duplicate count: {int(geolocation_clean.duplicated().sum())}")
print(f"geolocation columns: {geolocation_clean.columns} ")

geolocation shape: (738332, 5)
exact-duplicate count: 0
geolocation columns: Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='str') 


In [ ]:
# Count Geolocation records for each ZIP-code-prefix and state combination
geolocation_zipcode_state_combination = (geolocation_clean.groupby(['geolocation_zip_code_prefix', 'geolocation_state'])).size()
geolocation_zipcode_state_combination

geolocation_zip_code_prefix  geolocation_state
01001                        SP                   11
01002                        SP                    6
01003                        SP                   11
01004                        SP                   14
01005                        SP                   13
                                                  ..
99960                        RS                    5
99965                        RS                    6
99970                        RS                   16
99980                        RS                   21
99990                        RS                    2
Length: 19023, dtype: int64

In [ ]:
# Identify the most frequently recorded state for each Geolocation ZIP-code prefix
most_frequent_state_index  = geolocation_zipcode_state_combination.groupby('geolocation_zip_code_prefix').idxmax()
most_frequent_state_index

geolocation_zip_code_prefix
01001    (01001, SP)
01002    (01002, SP)
01003    (01003, SP)
01004    (01004, SP)
01005    (01005, SP)
            ...     
99960    (99960, RS)
99965    (99965, RS)
99970    (99970, RS)
99980    (99980, RS)
99990    (99990, RS)
Length: 19015, dtype: object

In [ ]:
# Extract the reference state from each selected ZIP–state combination
geolocation_zip_reference_state = most_frequent_state_index.apply(lambda x : x[1])
geolocation_zip_reference_state

geolocation_zip_code_prefix
01001    SP
01002    SP
01003    SP
01004    SP
01005    SP
         ..
99960    RS
99965    RS
99970    RS
99980    RS
99990    RS
Length: 19015, dtype: str

In [ ]:
# Map the dominant Geolocation state to each seller ZIP-code prefix

sellers_clean['geolocation_reference_state'] = sellers_clean['seller_zip_code_prefix'].map(geolocation_zip_reference_state)

In [ ]:
# Count sellers whose ZIP-code prefix has no Geolocation reference
sellers_clean['geolocation_reference_state'].isna().sum()

np.int64(7)

In [ ]:
# Inspect sellers whose ZIP-code prefix has no Geolocation reference
sellers_missing_geolocation_reference = sellers_clean[sellers_clean['geolocation_reference_state'].isna()]
sellers_missing_geolocation_reference

,seller_id,seller_zip_code_prefix,seller_city,seller_state,geolocation_reference_state
473,5962468f885ea01a1b6a97a218797b0a,82040,curitiba,PR,NaN
791,2aafae69bf4c41fbd94053d9413e87ee,91901,porto alegre,RS,NaN
1672,2a50b7ee5aebecc6fd0ff9784a4747d6,72580,brasilia,DF,NaN
1931,2e90cb1677d35cfe24eef47d441b7c87,02285,sao paulo,SP,NaN
2182,0b3f27369a4d8df98f7eb91077e438ac,07412,aruja,SP,NaN
2986,42bde9fef835393bb8a8849cb6b7f245,71551,brasilia,DF,NaN
3028,870d0118f7a9d85960f29ad89d5d989a,37708,pocos de caldas,MG,NaN


In [ ]:
# Flag sellers whose ZIP-code prefix has no Geolocation reference
flag_missing_geolocation_reference = sellers_clean['seller_id'].isin(sellers_missing_geolocation_reference['seller_id'])
sellers_clean['flag_missing_geolocation_reference'] = flag_missing_geolocation_reference
print(f"Confirming its data type: {sellers_clean['flag_missing_geolocation_reference'].dtype}")
print(f"count of true flag_missing_geolocation_reference values: {sellers_clean['flag_missing_geolocation_reference'].sum()}")

Confirming its data type: bool
count of true flag_missing_geolocation_reference values: 7


In [ ]:
sellers_clean

,seller_id,seller_zip_code_prefix,seller_city,seller_state,geolocation_reference_state,flag_missing_geolocation_reference
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,SP,False
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,SP,False
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,RJ,False
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,SP,False
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,SP,False
...,...,...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR,PR,False
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC,SC,False
3092,74871d19219c7d518d0090283e03c137,04650,sao paulo,SP,SP,False
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS,RS,False


In [ ]:
# Flag sellers whose recorded state differs from the available Geolocation reference state
flag_seller_state_mismatch = (sellers_clean['geolocation_reference_state'].notna()) & (sellers_clean['seller_state'] != sellers_clean['geolocation_reference_state'])
sellers_clean['flag_seller_state_mismatch'] = flag_seller_state_mismatch
print(f"Confirming its data type: {sellers_clean['flag_seller_state_mismatch'].dtype}")
print(f"count of true flag_seller_state_mismatch values: {sellers_clean['flag_seller_state_mismatch'].sum()}")

Confirming its data type: bool
count of true flag_seller_state_mismatch values: 35


In [ ]:
# Inspect sellers whose state differs from the dominant Geolocation reference state
seller_state_mismatches = sellers_clean[sellers_clean['flag_seller_state_mismatch'] == True]

In [ ]:
# Summarize seller-state and Geolocation-reference-state mismatch patterns
seller_state_mismatches_summary = seller_state_mismatches.groupby(['seller_state', 'geolocation_reference_state']).size().sort_values(ascending=False)
seller_state_mismatches_summary

seller_state  geolocation_reference_state
SP            PR                             10
              SC                              7
              MG                              6
              RJ                              4
              RS                              3
PA            PR                              1
RN            RJ                              1
SP            ES                              1
              BA                              1
              DF                              1
dtype: int64

In [ ]:
# Create a combined flag for sellers with any location-reference exception
flag_seller_location_exception = (sellers_clean['flag_missing_geolocation_reference'] == True) | (sellers_clean['flag_seller_state_mismatch'] == True)
sellers_clean['flag_seller_location_exception'] = flag_seller_location_exception
print(f"Confirming its data type: {sellers_clean['flag_seller_location_exception'].dtype}")
print(f"count of true flag_seller_location_exception values: {sellers_clean['flag_seller_location_exception'].sum()}")

Confirming its data type: bool
count of true flag_seller_location_exception values: 42


#### Seller Location-Consistency Findings

A dominant state reference was created for each Geolocation ZIP-code prefix using the most frequently recorded state. This approach avoids relying on the arbitrary first occurrence of a state.

Among the 3,095 sellers:

* **7 sellers** have ZIP-code prefixes that are absent from the Geolocation reference.
* **35 sellers** have a recorded state that differs from the dominant Geolocation reference state.
* **42 sellers** have at least one location-reference exception.

The affected seller records were retained and flagged rather than removed. These flags can be used to exclude or separately investigate the records when conducting state-level seller analysis.

### 8.6 Final Sellers Validation

The cleaned Sellers table will be validated to confirm that its original records were preserved, standardized fields have the expected formats, exception flags use Boolean data types, and seller relationships remain complete.


In [ ]:
# Confirm that Sellers cleaning preserved the original row count
if sellers_raw.shape[0] == sellers_clean.shape[0]:
    print("Both sellers_raw and sellers_clean df's have same rows")
    print(f"sellers_raw rows: {sellers_raw.shape[0]}")
    print(f"sellers_clean rows: {sellers_clean.shape[0]}")
else:
    print("sellers_raw and sellers_clean df's have different rows")
    print(f"sellers_raw rows: {sellers_raw.shape[0]}")
    print(f"sellers_clean rows: {sellers_clean.shape[0]}")

Both sellers_raw and sellers_clean df's have same rows
sellers_raw rows: 3095
sellers_clean rows: 3095


In [ ]:
#Confirm total number of columns in sellers_clean
len(sellers_clean.columns)

8

In [ ]:
#confirm Missing-value counts across all columns.
sellers_clean.isna().sum()

seller_id                             0
seller_zip_code_prefix                0
seller_city                           0
seller_state                          0
geolocation_reference_state           7
flag_missing_geolocation_reference    0
flag_seller_state_mismatch            0
flag_seller_location_exception        0
dtype: int64

In [ ]:
# Confirm that all seller location-exception flags use the Boolean data type
flags_created = ['flag_missing_geolocation_reference', 'flag_seller_state_mismatch', 'flag_seller_location_exception']
sellers_clean[flags_created].dtypes

flag_missing_geolocation_reference    bool
flag_seller_state_mismatch            bool
flag_seller_location_exception        bool
dtype: object

#### Final Sellers Validation Findings

All **3,095 seller records** were preserved during cleaning. The `seller_id` column remains complete and unique, and every seller referenced in the Order Items table exists in the Sellers table.

Seller ZIP-code prefixes were standardized as five-character strings, city values were trimmed and converted to lowercase, and state codes were standardized as two uppercase characters. No missing or blank values remain in the original Sellers columns.

The cleaned table contains seven missing Geolocation reference states, which correspond to seller ZIP-code prefixes absent from the Geolocation table. These records are documented by the missing-reference flag. All seller location-exception columns use the Boolean data type.

### 8.7 Save the Cleaned Sellers Table

The validated Sellers table will be exported to the processed-data directory. All original seller records and the newly created location-quality flags will be retained.


In [ ]:
# Save the validated Sellers table to the processed-data directory
sellers_clean.to_csv(Processed_Data_Path/'olist_sellers_clean.csv', index = False)

## 9. Geolocation Table Cleaning

This section prepares the Geolocation table for regional analysis and location validation. ZIP-code prefixes, city names, state codes, coordinates, and duplicate geographic observations will be standardized or flagged where necessary.

The `geolocation_clean` working copy was created while preparing the seller-location reference. Its ZIP-code prefixes and state codes have already been standardized, and 261,831 exact duplicate records were removed, leaving 738,332 unique geographic observations.

### 9.1 Geolocation City Standardization

Geolocation city values will be standardized by removing surrounding whitespace and converting the text to lowercase. Blank values and remaining whitespace inconsistencies will then be validated.


In [ ]:
# Standardize Geolocation city values by trimming whitespace and converting to lowercase

geolocation_clean['geolocation_city'] = geolocation_clean['geolocation_city'].str.strip().str.lower()

In [ ]:
# Validate standardized Geolocation city values
print(f"city blank or whitespace count: {int((geolocation_clean['geolocation_city'] == '').sum())}")
print(f"city leading space count: {int(geolocation_clean['geolocation_city'].str.startswith(' ').sum())}")
print(f"city trailing space count: {int(geolocation_clean['geolocation_city'].str.endswith(' ').sum())}")
print(f"number of distinct city values: {geolocation_clean['geolocation_city'].nunique()}")

city blank or whitespace count: 0
city leading space count: 0
city trailing space count: 0
number of distinct city values: 8010


#### Geolocation City Standardization Findings

No blank city values or surrounding whitespace remain after standardization. The number of distinct city values decreased from 8,011 to 8,010 because the city value containing trailing whitespace was normalized and consolidated with its existing equivalent.

### 9.2 Geographic Coordinate Validation

Latitude and longitude values will be checked against valid global coordinate limits and an approximate geographic boundary for Brazil. Records outside the expected Brazilian range will be retained and flagged as coordinate exceptions rather than removed.


In [ ]:
# Validate latitude and longitude values against global coordinate limits
print(f"Count of latitude values below -90 or above 90 : {len(geolocation_clean[(geolocation_clean['geolocation_lat']< -90)| (geolocation_clean['geolocation_lat']> 90)])}")

print(f"Count of longitude values below -180 or above 180 : {len(geolocation_clean[(geolocation_clean['geolocation_lng']< -180)| (geolocation_clean['geolocation_lng']> 180)])}")

Count of latitude values below -90 or above 90 : 0
Count of longitude values below -180 or above 180 : 0


In [ ]:
# Identify Geolocation records outside the approximate geographic bounds of Brazil
flag_coordinates_outside_brazil = ((geolocation_clean['geolocation_lat']< -34)| (geolocation_clean['geolocation_lat']> 6)) | ((geolocation_clean['geolocation_lng']< -74)| (geolocation_clean['geolocation_lng']> -32))
flag_coordinates_outside_brazil

0         False
1         False
2         False
3         False
4         False
          ...  
738327    False
738328    False
738329    False
738330    False
738331    False
Length: 738332, dtype: bool

In [ ]:
# Count Geolocation records outside the approximate geographic bounds of Brazil
int(flag_coordinates_outside_brazil.sum())

27

In [ ]:
# Add and validate the outside-Brazil coordinate exception flag
geolocation_clean['flag_coordinates_outside_brazil'] = flag_coordinates_outside_brazil
print(f" flag_coordinates_outside_brazil column datatype: {geolocation_clean['flag_coordinates_outside_brazil'].dtype} ")
print(f" flag_coordinates_outside_brazil column ture count: {int(geolocation_clean['flag_coordinates_outside_brazil'].sum())} ")

 flag_coordinates_outside_brazil column datatype: bool 
 flag_coordinates_outside_brazil column ture count: 27 


In [ ]:
# Count distinct ZIP-code prefixes affected by outside-Brazil coordinates
geolocation_outside_brazil = geolocation_clean[geolocation_clean['flag_coordinates_outside_brazil'] == True]
geolocation_outside_brazil['geolocation_zip_code_prefix'].nunique()

20

In [ ]:
# Retrieve all Geolocation observations for ZIP prefixes affected by coordinate exceptions
geolocation_zipcode_outside_brazil = geolocation_outside_brazil['geolocation_zip_code_prefix'].unique()
geolocation_affected_zip_records = geolocation_clean[geolocation_clean['geolocation_zip_code_prefix'].isin(geolocation_zipcode_outside_brazil)]
geolocation_affected_zip_records

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,flag_coordinates_outside_brazil
272046,18243,28.008978,-15.536867,bom retiro da esperanca,SP,True
355591,28165,41.614052,-8.411675,vila nova de campos,RJ,True
355597,28165,-24.576786,-53.795538,vila nova de campos,RJ,False
355603,28155,-34.586422,-58.732101,santa maria,RJ,True
355625,28165,-23.282553,-47.292447,vila nova de campos,RJ,False
...,...,...,...,...,...,...
732812,98780,-27.863935,-54.480055,santa rosa,RS,False
732815,98780,-29.937067,-51.025864,santa rosa,RS,False
732830,98780,-27.870662,-54.486393,santa rosa,RS,False
732832,98780,-27.863292,-54.471544,santa rosa,RS,False


In [ ]:
# Identify valid coordinate observations among the affected ZIP-code prefixes
latitude_within_brazil = ((geolocation_affected_zip_records['geolocation_lat'] >= -34) & (geolocation_affected_zip_records['geolocation_lat'] <= 6))
longitude_within_brazil = ((geolocation_affected_zip_records['geolocation_lng'] >= -74) & (geolocation_affected_zip_records['geolocation_lng'] <= -32))
valid_coordinates_for_affected_zips = geolocation_affected_zip_records[ latitude_within_brazil & longitude_within_brazil]

#verify number of records and distinct zip code prefix
print(f"Number of records present in Brazil boundary: {len(valid_coordinates_for_affected_zips)}")
print(f"Number of distinct zip code prefix: {valid_coordinates_for_affected_zips['geolocation_zip_code_prefix'].nunique()}")

Number of records present in Brazil boundary: 256
Number of distinct zip code prefix: 16


In [ ]:
# Identify affected ZIP-code prefixes with no valid coordinate observations
distinct_zip_valid_coordinates_for_affected_zips = valid_coordinates_for_affected_zips['geolocation_zip_code_prefix'].unique()
records_without_valid_coordinates = geolocation_outside_brazil[~geolocation_outside_brazil['geolocation_zip_code_prefix'].isin(distinct_zip_valid_coordinates_for_affected_zips)]
records_without_valid_coordinates

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,flag_coordinates_outside_brazil
272046,18243,28.008978,-15.536867,bom retiro da esperanca,SP,True
599205,78131,38.816816,-9.394625,varzea grande,MT,True
627844,83252,42.184003,-8.723762,ilha dos valadares,PR,True
710355,95130,14.585073,121.105394,santa lucia do piai,RS,True


In [ ]:
# Extract ZIP-code prefixes that have no valid coordinate observations
zip_prefixes_without_valid_coordinates = records_without_valid_coordinates['geolocation_zip_code_prefix'].unique()

# Confirm the number of ZIP-code prefixes without valid coordinate observations
len(zip_prefixes_without_valid_coordinates)

4

In [ ]:
# Flag Geolocation records belonging to ZIP prefixes with no valid coordinates
flag_zip_without_valid_coordinates = geolocation_clean['geolocation_zip_code_prefix'].isin(zip_prefixes_without_valid_coordinates)
geolocation_clean['flag_zip_without_valid_coordinates'] = flag_zip_without_valid_coordinates
print(f"flag_zip_without_valid_coordinates column datatype: {geolocation_clean['flag_zip_without_valid_coordinates'].dtype}")
print(f"Number of true flag_zip_without_valid_coordinates values: {geolocation_clean['flag_zip_without_valid_coordinates'].sum()}")

flag_zip_without_valid_coordinates column datatype: bool
Number of true flag_zip_without_valid_coordinates values: 4


#### Geographic Coordinate Findings

All latitude and longitude values fall within valid global coordinate limits. However, **27 records** fall outside the approximate geographic boundaries of Brazil, affecting **20 ZIP-code prefixes**.

Among these 20 affected ZIP prefixes, 16 have at least one alternative observation with valid Brazilian coordinates. The remaining **4 ZIP prefixes** have no valid coordinate observations. All records were retained and documented using Boolean exception flags rather than removed.

### 9.3 ZIP-to-State Consistency Validation

Some ZIP-code prefixes are associated with more than one state in the Geolocation table. Each record’s state will be compared with the dominant state for its ZIP prefix so that minority state assignments can be identified and flagged.


In [ ]:
# Map the dominant state reference to each Geolocation ZIP-code prefix
geolocation_clean['geolocation_reference_state'] = geolocation_clean['geolocation_zip_code_prefix'].map(geolocation_zip_reference_state)

In [ ]:
# Confirm that every Geolocation record received a reference state
geolocation_clean['geolocation_reference_state'].isna().sum()

np.int64(0)

In [ ]:
# Flag Geolocation records whose state differs from the dominant ZIP reference state
flag_geolocation_state_mismatch = geolocation_clean['geolocation_state'] != geolocation_clean['geolocation_reference_state']
geolocation_clean['flag_geolocation_state_mismatch'] = flag_geolocation_state_mismatch
print(f"flag_geolocation_state_mismatch column datatype: {geolocation_clean['flag_geolocation_state_mismatch'].dtype}")
print(f"Number of true flag_geolocation_state_mismatch values: {geolocation_clean['flag_geolocation_state_mismatch'].sum()}")

flag_geolocation_state_mismatch column datatype: bool
Number of true flag_geolocation_state_mismatch values: 8


In [ ]:
# Create a combined flag for records with any Geolocation quality exception
flag_geolocation_quality_exception = (geolocation_clean['flag_coordinates_outside_brazil'] == True) | (geolocation_clean['flag_zip_without_valid_coordinates'] == True) | (geolocation_clean['flag_geolocation_state_mismatch'] == True)
geolocation_clean['flag_geolocation_quality_exception'] = flag_geolocation_quality_exception
print(f"flag_geolocation_quality_exception column datatype: {geolocation_clean['flag_geolocation_quality_exception'].dtype}")
print(f"Number of true flag_geolocation_quality_exception values: {geolocation_clean['flag_geolocation_quality_exception'].sum()}")

flag_geolocation_quality_exception column datatype: bool
Number of true flag_geolocation_quality_exception values: 35


#### ZIP-to-State Consistency Findings

A dominant state was assigned to every ZIP-code prefix using the most frequently recorded state in the deduplicated Geolocation table. All Geolocation records received a reference state.

Eight records have a state value that differs from the dominant state associated with their ZIP-code prefix. These records were retained and flagged as state-assignment exceptions.

Overall, **35 Geolocation records** contain at least one quality exception:

* 27 records have coordinates outside the approximate geographic bounds of Brazil.
* 8 records have a state that differs from the dominant ZIP reference state.
* 4 of the 27 coordinate exceptions belong to ZIP prefixes with no valid alternative coordinate observations.

### 9.4 Final Geolocation Validation

The cleaned Geolocation table will be validated to confirm that exact duplicates were removed, standardized fields use the expected formats, exception flags use Boolean data types, and all remaining records are complete.


In [ ]:
# Confirm that the cleaned row count equals raw rows minus exact duplicates
raw_rows = geolocation_raw.shape[0]
duplicate_raw_rows = int(geolocation_raw.duplicated().sum())
clean_rows = geolocation_clean.shape[0]

print(f"raw row count: {raw_rows}")
print(f"number of exact duplicates originally identified: {duplicate_raw_rows}")
print(f"cleaned row count: {clean_rows}")
if (raw_rows - duplicate_raw_rows) == clean_rows:
    print(f"raw_rows - duplicate_raw_rows = clean_rows : {raw_rows - duplicate_raw_rows}")
else:
    print(f"raw_rows - duplicate_raw_rows != clean_rows : {raw_rows - duplicate_raw_rows}")

raw row count: 1000163
number of exact duplicates originally identified: 261831
cleaned row count: 738332
raw_rows - duplicate_raw_rows = clean_rows : 738332


In [ ]:
# Validate Geolocation columns, missing values, and remaining duplicate records
print(f"Total column count in geolocation_clean: {len(geolocation_clean.columns)}")
print(f"Missing-value counts across all columns: {geolocation_clean.isna().sum()}")
print(f"Remaining duplicates using only the five original Geolocation columns: {int(geolocation_clean[['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']].duplicated().sum())}")

Total column count in geolocation_clean: 10
Missing-value counts across all columns: geolocation_zip_code_prefix           0
geolocation_lat                       0
geolocation_lng                       0
geolocation_city                      0
geolocation_state                     0
flag_coordinates_outside_brazil       0
flag_zip_without_valid_coordinates    0
geolocation_reference_state           0
flag_geolocation_state_mismatch       0
flag_geolocation_quality_exception    0
dtype: int64
Remaining duplicates using only the five original Geolocation columns: 0


In [ ]:
# Validate Geolocation flag data types and standardized location formats
geolocation_flag_values = ['flag_coordinates_outside_brazil', 'flag_zip_without_valid_coordinates', 'flag_geolocation_state_mismatch', 'flag_geolocation_quality_exception']
print(f"geolocation flags data type: {geolocation_clean[geolocation_flag_values].dtypes}")

geolocation flags data type: flag_coordinates_outside_brazil       bool
flag_zip_without_valid_coordinates    bool
flag_geolocation_state_mismatch       bool
flag_geolocation_quality_exception    bool
dtype: object


In [ ]:
if min(geolocation_clean['geolocation_zip_code_prefix'].str.len()) == max(geolocation_clean['geolocation_zip_code_prefix'].str.len()):
    print("Both minimum and maximum length of zip code prefix are same")
    print(f"Minimum len of zip code prefix: {min(geolocation_clean['geolocation_zip_code_prefix'].str.len())}")
    print(f"Maximum len of zip code prefix: {max(geolocation_clean['geolocation_zip_code_prefix'].str.len())}")
else:
    print("Both minimum and maximum length of zip code prefix are different")
    print(f"Minimum len of zip code prefix: {min(geolocation_clean['geolocation_zip_code_prefix'].str.len())}")
    print(f"Maximum len of zip code prefix: {max(geolocation_clean['geolocation_zip_code_prefix'].str.len())}")

Both minimum and maximum length of zip code prefix are same
Minimum len of zip code prefix: 5
Maximum len of zip code prefix: 5


In [ ]:
#Confirm that geolocation_reference_state contains two-character uppercase codes
geolocation_reference_state_not2_notupper_count = ((geolocation_clean['geolocation_reference_state'].str.len() != 2) | (~geolocation_clean['geolocation_reference_state'].str.isupper()))
int(geolocation_reference_state_not2_notupper_count.sum())

0

#### Final Geolocation Validation Findings

The cleaned Geolocation table contains **738,332 unique records** after removing **261,831 exact duplicate rows** from the original 1,000,163 records.

All original columns remain complete, and no exact duplicate observations remain. ZIP-code prefixes are consistently stored as five-character strings, city values have been standardized, and state values use two-character uppercase codes.

The table contains four Boolean data-quality flags covering coordinates outside Brazil’s approximate boundaries, ZIP prefixes without valid coordinate alternatives, state assignments that differ from the dominant ZIP reference state, and the combined Geolocation quality exception.

The identified exceptions were retained and flagged so they can be reviewed or excluded from geographic analysis without losing the original observations.

### 9.5 Save the Cleaned Geolocation Table

The validated and deduplicated Geolocation table will be exported to the processed-data directory with all quality flags and the dominant state reference retained.


In [ ]:
# Save the validated Geolocation table to the processed-data directory
geolocation_clean.to_csv(Processed_Data_Path/"olist_geolocation_clean.csv", index=False)

## 10. Cleaning and Preparation Summary

All nine Olist datasets were cleaned, validated, and exported to the processed-data directory:

* Orders
* Customers
* Order Items
* Order Payments
* Order Reviews
* Products
* Product Category Translation
* Sellers
* Geolocation

The cleaning process standardized timestamp, ZIP-code, city, state, and category fields; corrected product column names; removed exact Geolocation duplicates; and created Boolean flags for identified data-quality exceptions.

Valid records were preserved wherever possible. Missing values and unusual records were retained when they represented expected business conditions or isolated exceptions, allowing later analyses to include or exclude them according to the metric being calculated.

The processed datasets are now ready to be combined into an analysis-ready order-item-level dataset.
